# Earth Observation Foundation Models: Encoder Representation Analysis and Downstream Task Benchmarking

In [1]:
from google.colab import drive
drive.mount('/content/drive')

!cd /content/
#!git clone --branch v0.1 --single-branch https://github.com/sig-gis/pangaea-bench.git
!git clone --branch eofm_book  --single-branch https://github.com/sig-gis/pangaea-bench.git #TODO re-tag and update clone
!pip install -e "pangaea-bench"
!pip install -r pangaea-bench/requirements.txt
!git clone --branch v0.1 --single-branch https://github.com/sig-gis/calculate-flops.pytorch.git
#Replate <absolute_path_to> with the absolute path to the calflops repository you just cloned
!export PYTHONPATH=${PYTHONPATH}:/content/calculate-flops.pytorch/


Mounted at /content/drive
Cloning into 'pangaea-bench'...
remote: Enumerating objects: 5151, done.
remote: Counting objects: 100% (1316/1316), done.
remote: Compressing objects: 100% (467/467), done.
remote: Total 5151 (delta 1074), reused 860 (delta 849), pack-reused 3835 (from 2)
Receiving objects: 100% (5151/5151), 3.66 MiB | 10.18 MiB/s, done.
Resolving deltas: 100% (3511/3511), done.
Obtaining file:///content/pangaea-bench
  Preparing metadata (setup.py) ... done
  Running setup.py develop for pangaea
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 3.1 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of rasterio to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of tifffile to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of opencv-python to determine which version is compatible with other require

Cloning into 'calculate-flops.pytorch'...
remote: Enumerating objects: 304, done.
remote: Counting objects: 100% (112/112), done.
remote: Compressing objects: 100% (18/18), done.
remote: Total 304 (delta 101), reused 94 (delta 94), pack-reused 192 (from 1)
Receiving objects: 100% (304/304), 3.62 MiB | 9.36 MiB/s, done.
Resolving deltas: 100% (202/202), done.
Note: switching to 'a2e3059add9a91458c69bf6831ac55a62b2fa9c1'.

You are in 'detached HEAD' state. You can look around, make experimental
changes and commit them, and you can discard any commits you make in this
state without impacting any branches by switching back to a branch.

If you want to create a new branch to retain commits you create, you may
do so (now or later) by using -c with the switch command. Example:

  git switch -c <new-branch-name>

Or undo this operation with:

  git switch -

Turn off this advice by setting config variable advice.detachedHead to false



In [1]:

#CROMA weights and datasets are large and sometimes fail to download automatically. Ensuring we have them by manually downloading here.
!mkdir -p /content/pangaea-bench/pangaea/encoder_analysis/pretrained_models/
!mkdir /content/data/
!ln -s /content/data/ /content/pangaea-bench/pangaea/encoder_analysis/

!wget https://huggingface.co/antofuller/CROMA/resolve/main/CROMA_large.pt -O /content/pangaea-bench/pangaea/encoder_analysis/pretrained_models/CROMA_large.pt

!wget https://huggingface.co/datasets/ibm-nasa-geospatial/hls_burn_scars/resolve/main/hls_burn_scars.tar.gz -O /content/data/hls_burn_scars.tar.gz
%pushd /content/data/
!mkdir ./hlsburnscars/
!tar -xf hls_burn_scars.tar.gz -C ./hlsburnscars/
%popd



--2026-06-08 05:47:14--  https://huggingface.co/antofuller/CROMA/resolve/main/CROMA_large.pt
Resolving huggingface.co (huggingface.co)... 108.138.246.71, 108.138.246.79, 108.138.246.67, ...
Connecting to huggingface.co (huggingface.co)|108.138.246.71|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://cas-bridge.xethub.hf.co/xet-bridge-us/654aab72f8ebcec5451bd372/ae5b85ce5496162e7524f76b15f447531cdf9c77f656fc49a106c74003099405?Expires=1780901234&Policy=eyJTdGF0ZW1lbnQiOlt7IlJlc291cmNlIjoiaHR0cHM6Ly9jYXMtYnJpZGdlLnhldGh1Yi5oZi5jby94ZXQtYnJpZGdlLXVzLzY1NGFhYjcyZjhlYmNlYzU0NTFiZDM3Mi9hZTViODVjZTU0OTYxNjJlNzUyNGY3NmIxNWY0NDc1MzFjZGY5Yzc3ZjY1NmZjNDlhMTA2Yzc0MDAzMDk5NDA1KiIsIkNvbmRpdGlvbiI6eyJEYXRlTGVzc1RoYW4iOnsiQVdTOkVwb2NoVGltZSI6MTc4MDkwMTIzNH19fV19&Signature=MEUCIQDADvaMTlMpX4v3DgG1Nk2kyiyT7X2SO-XvMSnkr8J27AIgRJS0kiUCn274crs2zrC%7EhmkhJFxtca%7E-agK1jw4XzKM_&Key-Pair-Id=K1LYXO563TGWFU&X-Xet-Cas-Uid=public&response-content-disposition=inline%3B+filename*

In [3]:
#Move to directory.
#This directory will be used for all subsequent encoder analysis tasks as well
%pushd /content/pangaea-bench/pangaea/encoder_analysis/

#Library writes to local directories.
!mkdir img
!mkdir ww-img
!mkdir -p ww/croma/
!mkdir -p ww/dofa/
!mkdir -p ww/prithvi/
!mkdir ./embeddings/


/content/pangaea-bench/pangaea/encoder_analysis


## Section A: Introduction

---
### Scientific Background and Problem Formulation
---

In this notebook we will be exploring adapting Earth Observation Foundation Models (EOFMs) to a pair of wildfire monitoring tasks. To begin, we will discuss the tasks, the associated challenges, and the utility that emerging techniques may have to offer. Our goal in this chapter is to walk through these ideas step by step, assuming only a basic familiarity with machine learning and remote sensing.

#### Problem Formulation

Following a wildfire event, it is vital to assign resources to determine the extent of the damages caused to people, their properties, and the environment, and the depth of the long-term effects of the event. This can be done through examining remote sensed Earth observation data, such as Sentinel-2 imagery. However, these assessments can be time-consuming and expensive to produce.

The work is traditionally done using hand-tuned thresholds for specific indices such as NBR (Normalized Burn Ratio) and dNBR (delta Normalized Burn Ratio) (Cocke et al., 2005; Roy et al. 2005). These thresholds are then specific to the regions they are developed for, sometimes even specific to a single event, and require expert knowledge to produce. This leaves these assessments inaccessible to many that lack the resources and access to expertise required for their generation, creating many barriers which make the prospect of automation attractive for these tasks.

More traditional data-driven approaches, such as supervised learning methods, may still impose significant requirements of large labeled datasets and custom architecture development. With EOFMs, pretrained using self-supervision, comes the hope that models will learn generalizable features from large collections of unlabeled data whose insights can then be transferred to a wide array of specific tasks. This is connected to the hope that fine-tuning these models will be more straightforward than traditional methods, thereby saving on dupications of effort and costly model training time.

Here we will examine two tasks related to the post-wildland-fire severity assessment context described above: mapping burn scars and mapping burn severity. In both cases, we work with optical satellite imagery and associated labels. The datasets we will use are:

- HLS Burn Scars: The HLS Burn Scars dataset (Phillips et al., 2023) consists of paired Harmonized Landsat–Sentinel (HLS) images and corresponding burn scar masks from 2018–2021 over the contiguous United States. The input imagery is continuous reflectance data across multiple spectral bands, and the labels are binary masks indicating burned vs. unburned pixels for each scene. These masks are derived for individual fire events. This structure makes the dataset well suited for semantic segmentation and for studying how encoders separate burned and unburned classes in embedding space.

- Monitoring Trends in Burn Severity (MTBS): The MTBS project uses expert-driven interpretation of pre‑ and post‑fire imagery to assign categorical burn severity classes (e.g., unburned/low, moderate, high) within mapped fire perimeters. In our setup, we pair these discrete severity labels with Landsat-based optical imagery collected before and after fire events. As with the HLS Burn Scars dataset, the input data are continuous reflectance values, but here the targets are multi-class severity labels. The temporal baseline is again event-centered: images are selected relative to each fire’s occurrence to capture pre‑ and post‑fire conditions. This combination of continuous imagery and categorical severity classes provides a way to test how different encoders structure severity gradients in their embedding spaces, and how well various decoders can recover those distinctions.

As we make decisions on utilizing and adapting existing foundation models to these tasks, many questions arise, including:

- What do we need to do to adapt our EOFMs to these tasks? What are the downstream impacts of these decisions on performance and usability of our final solution?
- Which models perform the best on our task? How do we compare them to each other and to our possible baseline methods?

Underlying all of these practical questions are more abstract questions regarding the validity of these models and why they behave the way they do. We will discuss these concepts in detail in this tutorial, using the aforementioned post-fire monitoring problems as a frame in order to assist the reader in developing understanding of these topics, and apply them to their use cases. Because EOFMs are still rapidly evolving, many topics we touch on remain open research questions, and the material here will continue to change as the associated fields progress. We aim to provide a snapshot of the situation as it stands as well as commentary to help frame discussion on approaching these open topics in order to equip our readers to confront them.

To do that, we first introduce a few core ideas from modern deep learning, including transfer learning, foundation models, and embeddings, and connect them to the wildfire tasks described above.


---
### Technical Background and Introduction
---

#### Transfer Learning and Beyond

---

Transfer learning is a powerful technique in machine learning that allows a model trained on one task to leverage previously learned features for a different, but related task. Instead of starting from scratch, transfer learning leverages the knowledge a model has already gained from a separate, and usually large, dataset and applies it to a new problem. This is particularly useful when a dataset is small or annotations for the new task are limited. Once an initial training has been completed, the layers towards the front (closer to the input layer) of these models have typically learned information about general features, such as edges in images or syntactic patterns in text, which can be useful across a wide range of tasks.

Ideally, by fine-tuning the initial layers of a pretrained model or adding new layers specific to the new task, practitioners can adapt it to perform well with far fewer training samples. This also has the potential to improve performance thresholds for a given smaller or imbalanced dataset, as the pretrained model already has latent feature extraction capabilities from prior training.  Properly executed transfer learning not only has the potential to boost performance, but also saves computational resources and significantly reduces training time. You can think of this as skill reuse: a model that has already learned to recognize generic shapes and textures in one dataset can adapt more quickly to a new dataset.


Many different strategies can be applied to achieve transfer learning, such as freezing different sets of layers during training, or allowing all layers to be updated, depending on the similarity between the source (initial) and target (new) tasks. For example, in medical imaging, models trained on natural images have been fine-tuned to detect tumors or classify X-rays by retraining only a few layers.

<img style="display: block;
           margin-left: auto;
           margin-right: auto;
           width: 60%;"
      src="https://raw.githubusercontent.com/sig-gis/eofm-book/main/assets/pretraining_workflow.png"/>

<div style="text-align: center;"> Figure 1. Simplified foundation model pretraining workflow</div>
<br/>

An emerging category of models, termed **'Foundation Models' (FMs)**, use transfer learning at a much larger scale. In these models, the pretraining task is designed to learn general-purpose representations from a large unlabeled collection of data. These representations then can serve as the **foundation** for many different models and downstream tasks (Figure 1). This is in contrast to traditional supervised learning problems that aim to map data samples to a annotations from a specific task. FMs are said to be trained in a 'self-supervised' manner and potentially offer similar benefits to models developed using traditional transfer learning techniques (i.e reduced training time and performance). However, unlike with transfer learning, FMs circumvent the development of an initial large scale labeled dataset. This technique is especially powerful in the remote sensing world where the availability of large scale unprocessed data is ever-increasing, and the methods for transforming it into actionable insights draw from a wide range of scientific disciplines. **The larger goal of the EOFM development communitity is to create models capable of generalizing across Earth Science domains and remote sensing modalities by learning from the structure and correlations inherent in the raw data, and enabling a wider array of practitioners, and more versatile models to adapt them with minimal fine-tuning. Attempts towards this ephemeral goal, however, must be thoroughly assessed in a way that adheres not just to typical computer vision analysis, but also to the rigor required with respect to both the underlying science goals and the remote sensing instrumentation** (DeMilt et al., 2025).

If truly foundational, a foundation model plays a similar role to a good set of “starter features” that many different teams can build on. Instead of each project training a model from scratch, a single, large encoder can be adapted to many downstream tasks, including regression, segmentation, and change detection, by adding relatively small task-specific components on top. We refer to these small supplementary models as "decoder heads". Their job is to customize our pretrained foundation model to complete our task.

In the next few subsections, we briefly survey how these encoders are pretrained. It is not required to master all of the details of each method, but we want to provide enough context to help build intuition and depth of interpretation for use later in the chapter.

---

####  Pretraining Methodologies

---

Below are three categories of methods for pretraining a computer vision model in a self-supervised fashion. These methods have been combined and extended in many ways to create the present field of EOFMs. This list is not exhaustive, as widespread satellite data access has led to many variations and combinations, as well as other techniques that are less commonly used. There is also an extensive effort to interconnect strides made separately within the computer vision and natural language processing domains both in and out of the remote sensing community.

It is worth noting that the models described and tested here are all transformer-based, where the core architectural unit is a vision-transformer trained under the 'encoder-decoder' paradigm (to be discussed). While it is important to understand the high level distinction between these techniques, it is not entirely necessary in order to implement them in practice as many developer groups have worked hard in abstracting most of the logic away in various frameworks (Han et al. 2022). Deep understanding of each of these model families, as well as the nuances of transformer architectures, is beyond the scope of this chapter, and not strictly necessary to benefit from the material here. We encourage interested readers to explore these topics further, but our focus will be on the main ideas and how to use them in practice. The deeper a practitioner's knowledge, the easier it will be to distinguish between appropriate and inappropriate practices both with data and the associated model training and deployment.

#### <u>Masked Autoencoding (MAE)</u>

<img style="display: block;
           margin-left: auto;
           margin-right: auto;
           width: 50%;"
      src="https://raw.githubusercontent.com/sig-gis/eofm-book/main/assets/mae.png"/>

<div style="text-align: center;"> Figure 2. Diagram depicting masked autoencoder training from (He et al., 2022). Images are broken into small sections, called patches. A percentage of these patches are hidden (masked). The decoder is tasked with reconstructing the original image from the remaining unmasked patches. </div>
<br/>

Autoencoder-centric methods consist of an encoder that projects input images into a (typically) lower-dimensional latent (or embedding) space and a decoder that reconstructs the original images from these representations. This compression helps the model capture essential features while discarding noise or irrelevant details. By reconstructing lost information with remaining unmasked patches, models are expected to encode general features of images and develop an image-level understanding and fine-grained spatial reasoning. This process is sometimes referred to as masked image modeling (MIM). Masked tokens do not have to be processed by the encoder, as they are effectively ignored. Higher masking levels therefore reduce computation per sample, which can be traded for more training iterations or larger datasets (He et al., 2022). Figure 2 depicts the masked autoencoder training flow.

#### <u>Contrastive Learning</u>

<img style="display: block;
           margin-left: auto;
           margin-right: auto;
           width: 50%;"
    src="https://raw.githubusercontent.com/sig-gis/eofm-book/main/assets/simclr.png"/>

<div style="text-align: center;"> Figure 3. Diagram for SimCLR training paradigm from (Chen et al., 2020). </div>
<br/>

Contrastive models are tasked with distinguishing between similar and dissimilar image representations. They do this by pulling together representations of positive pairs (different augmented views of the same sample) so that they lie close to one another in the model's embedding space, while pushing dissimilar pairs farther apart, typically using views from different images. An example flow diagram can be seen in Figure 3. Common methods include Simple Contrastive Learning (SimCLR) and Momentum Contrast (MoCo) (Chen et al., 2020; He et al., 2020).

In this setting, and the broader context of deep learning, an embedding is a vector representation produced by the encoder for each sample (here, an image or image patch). Informally, you can think of an embedding as the model's internal summary of an input sample: a list of values that captures which patterns, colors, and shapes it considers most important for distinguishing one scene from another. The contrastive loss operates directly on these vectors or lists, shaping the geometry of the latent space so that samples deemed similar by the training signal occupy nearby regions, and dissimilar samples are separated. Unlike with masked autoencoders, a decoder is typically not included in the pretraining phase, since the task is defined entirely in this latent space rather than in the pixel space of the original image.

This methodology usually requires some level of annotation or at least a notion of correspondence, for example by defining positive and negative pairs within the training data, and can sometimes be referred to as a semi-supervised methodology. The resulting training objective tends to produce embedding spaces with stronger, more explicit notions of similarity: clusters or manifolds that reflect semantic or physical relationships between samples. This, in turn, leads to latent space properties that can differ substantially from those of masked autoencoder models.



#### <u>Non-contrastive Self Distillation</u>

<img style="display: block;
           margin-left: auto;
           margin-right: auto;
           width: 50%;"
    src="https://raw.githubusercontent.com/sig-gis/eofm-book/main/assets/dino.png">
</img>

<div style="text-align: center;"> Figure 4. Diagram for DINO training paradigm from (Caron et al., 2021). </div>
<br/>

Non-contrastive self-distillation is a self-supervised learning approach for computer vision that avoids the need for negative samples. Instead of contrasting different images, it trains a student network to match the output of a teacher network, both fed with different augmented views of the same image. The teacher is often an exponential moving average of the student, meaning that it is a slowly changing average of student models. This provides stable targets for the two encoders while distilling knowledge from the teacher model. Figure 4 shows an example of this approach. The objective is to produce consistent, meaningful embeddings, not to reconstruct or generate the input. Common methods include 'Self-**di**stillation with **no** labels' (DINO) and 'Bootstrap Your Own Latent' (BYOL) (Grill et al., 2020; Caron et al., 2021). These methods are often combined with different pretraining objectives and are a strong method for refining new models while utilizing the results of existing ones.

---

#### Decoders

---

Just as you would with any trained neural network, you can load the weights for inference using similar code you might have seen in previous chapters. There is still one more consideration you must make depending on the type of task at hand and the shape of the output of the encoder. If your goal is to classify images, a lightweight machine learning algorithm directly on top of the embedding vector outputs of the encoders may be sufficient. The assumption here is that the model's internal representation of the image is robust and **relevant** to your task. We will touch on this topic below. If your goal, on the other hand, is to segment the images, we may need to train a subsequent neural network to extract from that internal representation and produce a fine-grained map. Below is a summarization of contemporary decoder types commonly used in conjunction with transformer-based encoders. Each varies in complexity, purpose, and method in which they extract features from the transformer layers in the aforementioned encoders.

#### Linear Probe

<img style="display: block;
           margin-left: auto;
           margin-right: auto;
           width: 70%;"
           src="https://raw.githubusercontent.com/sig-gis/eofm-book/main/assets/linear.png"/>

<div style="text-align: center;">Figure 5. Simplified structure of a single linear layer.</div>
<br/>

A linear probe, as displayed in Figure 5, is the simplest type of decoder typically used in this paradigm. It is a single linear layer, or small multi-layer perceptron (MLP). This technique is taken from the language domain where large language models are often evaluated on downstream tasks with linear probes on frozen embeddings before doing full fine-tuning.  In practice, this isn't typically used for many Earth observation tasks due to limited model complexity. But this decoder is still useful in assessing representation quality as it is cheap to train and can be implemented consistently, making it an ideal tool for benchmarking. Since the model is so simple, the performance of a linear decoder can be said to give an analysis of how easily separable the encoder has made the samples from our dataset.

##### Fully Convolutional Network (FCN)

<img style="display: block;
           margin-left: auto;
           margin-right: auto;
           width: 60%;"
           src="https://raw.githubusercontent.com/sig-gis/eofm-book/main/assets/fcn.png"/>

<div style="text-align: center;"> Figure 6. Full convolutional neural network architectural diagram from (Long et al., 2014).</div>
<br/>

The FCN, shown in Figure 6, is a stack of convolutional layers that upsamples and refines final feature maps produced by the transformer’s output embeddings and converts them back into an image-like spatial representation. This final representation can then be fed through a final prediction layer, which is typically itself convolutional layers, to produce the final map (Long et al., 2014). This is typically more lightweight than some of the MLP-based decoders below, due to weight sharing in the learned convolutional kernels. Local context is also explicitly emphasized as those same kernels typically only cover a small portion of the whole input image at a given time.

##### SegFormer

<img style="display: block;
           margin-left: auto;
           margin-right: auto;
           width: 60%;"
           src="https://raw.githubusercontent.com/sig-gis/eofm-book/main/assets/segformer.png"/>

<div style="text-align: center;"> Figure 7. SegFormer style network architectural diagram for semantic segmentation from (Xie et al., 2021).</div>
<br/>

SegFormers use an MLP style decoder to extract information from a set of hierarchical features taken from transformer-based encoders (Xie et al., 2021). An example is shown in Figure 7. Originally developed as a standalone framework for end to end training, the SegFormer has been recently adapted by a handful of research groups attempting to utilize the representation generated by the pretraining phase. The assumption is that the implementation of the transformer blocks in the encoder are sufficiently hierarchical in nature such that they can be composed using this method.

##### UperNet

<img style="display: block;
           margin-left: auto;
           margin-right: auto;
           width: 60%;"
           src="https://raw.githubusercontent.com/sig-gis/eofm-book/main/assets/upernet.png"/>

<div style="text-align: center;">Figure 8. UperNet architectural diagram from (Xiao et al., 2018).</div>
<br/>

Similar to the SegFormer architecture, the UperNet leverages multi-scale hierarchical features to generate dense (per-pixel) predictions. Rather than MLPs, the UperNet utilizes pooling layers and convolutions to compose the features before the final prediction layers. An example of UperNet can be seen in Figure 8. It's worth noting here that the original UperNet use case used a ResNet-50 backbone that was fine tuned. In other words, this architecture has been closely aligned with the 'pretrained encoder' / decoder paradigm that has been popularized in the last few years (Xiao et al., 2018).

##### MUSTER

<img style="display: block;
           margin-left: auto;
           margin-right: auto;
           width: 60%;"
           src="https://raw.githubusercontent.com/sig-gis/eofm-book/main/assets/muster.png"/>

<div style="text-align: center;">Figure 9. Architectural diagram of MUSTER decoder from (Xu et al., 2022).</div>
<br/>

MUSTER, as seen in Figure 9, is a more recent decoder development in the computer vision realm. Similar to both the SegFormer and the UperNet. Hierarchical features play an important role in it's predictive strength with the most important distinction here being that the decoder itself is also transformer based (Xu et al., 2022). Also similar to the UperNet, the MUSTER decoder was designed to integrate with pretrained encoders. While this may lend itself to greater performance, it is important to consider the compute required to implement such an architecture.


---

#### Encoder Evaluation

---


When working with EOFMs, it is not enough to evaluate only the final task performance. We also need to understand how well the encoder itself represents the data. This process requires a multifaceted approach on both the data and analysis fronts. As an encoder model is meant to have broad purpose and applicability, no single metric or dataset is going to be sufficient to understand the performance and utility of the model for its downstream users. For a beginner, it may help to think of this as stress-testing the encoder: on top of understanding if a model simply performs well on a given task or benchmark, we want to understand how it behaves under different data conditions, what kinds of patterns it tends to learn, and what is the computational and environmental cost of training and deploying various solutions.

Despite this complexity there are common factors to consider. Diversity of datasets, which can cover broad conceptual categories within the fields of remote sensing and Earth system science, is an important factor. While no dataset is the perfect exemplar, a dataset similar to the one a given model was pretrained on dataset may give an initial assessment of a model's applicability to a new task. Diverse training and inference settings are equally important. Understanding a model's performance in many settings such as varying image sizes, geographic regions, and spatial scales can reveal strengths and weaknesses of each encoder and can enlighten both the downstream user and the researcher looking to improve their encoder.

While this chapter is not focused on dataset accumulation, it should also be mentioned that proper splitting of the data in a geospatial setting is imperative for proper evaluation and should consider variables like keeping unseen spatiotemporal areas as a held-out test set for independent post-training evaluation of a model. This ensures that complex spatiotemporal cues built into geospatial datasets are not "memorized" and overstate a model's performance on a given task and potential generalizability (LaHaye et al., 2021; Parajuli et al., 2024). Here we discuss some ways an encoder's representational "skill" can be measured and evaluated by looking at both the weight matrix of the model and the model's embedding vectors. Unlike the metrics used for downstream task evaluation, many of these are newer and therefore less known. Here we provide a brief description and references for each methodology used.

##### <u>Resource Requirements</u>

In this section, we introduce simple proxies for resource requirements of a given model (Floating Point Operations (FLOPs), Multiply-Accumulate Operations (MACs), and parameter counts) that help you estimate whether a model will fit your hardware and runtime constraints. As data science practitioners, we typically want to balance performance, interpretability, and resource requirements as three main driving factors in model selection. It is also crucial that we aim to choose the smallest and most efficient model that meets the requirements for our task in these three areas. This is important not only for the sake of efficiency, but also because the environmental impact of large-scale deep learning models has become a significant concern as model sizes and computational demands continue to grow. Choosing small and optimized models where appropriate is an effective strategy for reducing the emissions and energy consumption footprint associated with model development and deployment (Strubell et al., 2019; Patterson et al., 2021) . These considerations should also include a model's pretraining phase as well as resources required for finetuning and operational inference.


FLOPs, MACs, and parameter counts are three metrics that offer practical proxies for estimating a model's resource requirements. FLOPs denote the total number of floating-point operations (additions, multiplications, etc.) required to process a single input through the model. High FLOPs indicate a model that requires substantial computation resources, which can translate directly to longer inference and training times. Multiply-accumulate operations, foundational to neural network layers (especially in convolutions and matrix multiplications) are closely linked to the actual physical operations that deep learning accelerators (like GPUs and TPUs) execute, thus providing an estimate of real computational workload. The parameter count measures the total number of learnable variables (weights and biases) within a model. The number of parameters is a large driver for the memory required to load and store the model during execution and on disk.

Some limitations that should be considered:

1) FLOPs are platform-agnostic and do not directly account for hardware-specific optimizations or parallelization capabilities.

2) MACs are specifically meaningful for models dominated by linear algebra; other operations (e.g., activations, normalizations) may not be included in MAC counts

These parameters collectively can give us a well-rounded glimpse at model resource requirements, and allow us to factor that in when doing intercomparisons.


##### <u>Weight Matrix Analysis for a Data-Free Assessment of Training Quality</u>

The Weight Watcher analysis library introduces the measure of an implicit form of self-regularization, as revealed by the spectral properties of a model's weight matrices (Martin et al.,  2021). The toolkit and associated papers apply Random Matrix Theory (RMT), a branch of mathematics that studies the properties of matrices whose entries are random variables. More specifically Empirical Spectral Density (ESD) analysis is used. This is the characterization of the distribution of eigenvalues of a given matrix, often large, random, or structured matrices forming a histogram that represents how the eigenvalues are distributed across their range of possible values. As part of this analysis, we summarize each layer's ESD using the power-law exponent α $\alpha$, which characterizes the tail of the distribution.

When the distribution of singular values of a weight matrix follows a power law, ESD behaves as $P$($\lambda$)∼$\lambda^{-\alpha}$, where $\lambda$ denotes eigenvalues and $\alpha$ is the power-law exponent. $\alpha$ quantifies how heavy-tailed, or spread out, the layer's eigenvalue distribution is. Smaller $\alpha$ values correspond to heavier tails, indicating the weight matrix has more large singular values, mapping to more complex structure and a higher risk of fitting noise as well as signal (overfitting). Larger $\alpha$ values correspond to faster-decaying tails, indicating more noise-like randomness or over-regularization, where the layer may fail to capture important data structure (underfitting). Moderate $\alpha$ values are interpreted as a sign of useful complexity without excessive overfitting in practice.

Here we provide the means to do the full ESD-based analysis that the WeightWatcher tool provides, and use the. $\alpha$ parameter as a summarization metric for the analysis.


##### <u>Working with Embeddings</u>

The vector representation constructed by processing a sample with an encoder is often called an embedding, and the space where all these embeddings are drawn from is known as the embedding space. One of the most promising prospects of generalizable learned features from EOFMs is the ability to reason over their embeddings. The goal of an embedding is to synthesize important information, and to provide abstract features which can be used to more easily distinguish similar and dissimilar samples under more diverse contexts than can be done with the raw input data alone. Embedding-based approaches are currently being explored and utilized for similarity search, few/zero shot learning, and low label monitoring tasks.

However, these spaces are high dimensional (100s or 1000s of dimensions for each patch) and are the result of the learned function of our neural network and thus there is no prima facie method for understanding the meaning or importance of placement in this latent space. Thankfully, much work has been done on tools for exploring these spaces and investigating for features or distinctions of interest within them. These include longstanding visualization techniques such as UMAP (explored below) and newer work investigating comparisons between multiple models.

##### <u>Manifold Projections for Qualitative Analysis and Dimension Reduction</u>

The Uniform Manifold Approximation and Projection (UMAP) methodology for Dimension Reduction can be used both as a general dimension reduction tool, as an interpretability / visualization tool, and here we use it for both. UMAP assumes data lies on a manifold and utilizes local approximations (k-nearest neighbor graphs) to represent the data's topological and metric structure. Each local neighborhood is encoded as a fuzzy simplicial set, and then merged probabilistically to form a global topological structure (McInnes et al., 2020). Figure 10 provides a couple of examples of embeddings from different models and geospatial datasets projected into a lower-dimension feature space using UMAP, and overlayed with each sample's target or label value.


<img style="display: block;
           margin-left: auto;
           margin-right: auto;"
           src="https://raw.githubusercontent.com/sig-gis/eofm-book/main/assets/UMAP_Examples.png"/>
<div style="text-align: center;"> Figure 10. Embeddings from different models and geospatial datasets with 4 classes (left), 2 classes (center), and 7 classes (right) projected into a 2D feature space using UMAP. </div>
<br/>
Notice that the separation of label sets in the projections degrades from left to right, indicating that the models increasingly struggle to represent class distinctions in a way that is easily recoverable in a low-dimensional space. This does not always mean that a model is fundamentally unable to represent the underlying data structure; in some cases, the representation may simply be more complex and non-linear, which makes faithful projection into two dimensions difficult. That limitation is important to keep in mind when interpreting these plots, but the analysis remains useful in many ways. The primary two points of utility in this context are described below.

First, complex representations leading to poor or muddled separation in the UMAP-projected space often signal that downstream learning tasks will be more challenging. A decoder trained on such embeddings must work harder to tease apart classes, may require more capacity or careful regularization, and can be more sensitive to hyperparameters and data preprocessing. Conversely, when classes form clean, well-separated clusters in these projections, it suggests that relatively simple decoders may be sufficient, which can guide architectural and training choices.

Second, these projections provide a bridge between model behavior and human interpretability. **In scientific applications, our aim is not only to achieve high predictive performance but also to understand why a model behaves the way it does and how its representations relate to known processes.** Visualizing how embeddings cluster, or fail to cluster, by class, region, or other attributes allows domain experts to assess whether a model organizes information in ways that are scientifically meaningful, to spot systematic confusions, and to generate new hypotheses. In that sense, manifold projections are both a diagnostic tool for model selection and a vehicle for scientific insight, which we view as essential for data science and deep learning applications in these domains.

Some additional limitations include:

1) the use of stochastic operations leading to different runs or subsampling yielding varied results and
2) the quality and nature of embeddings are sensitive to hyperparameters, although empirically, the authors note that while this is true a general similarity of structure can be achieved relative to class separation if the representations of N classes are distinct enough.

While useful, the uncertainties of these tools require us to use multiple approaches to get a more comprehensive look at the state of model representation. To complement these qualitative projections, we next turn to data-kernel-based methods that quantify representational differences between encoders in a more statistically rigorous way.


##### <u>Data Kernels for Inter-Model Embedding Comparisons</u>

Encoder projections and embedding geometries are not directly comparable due to inherent incongruities - namely, differences in the dimensionality of original encodings or in their basis vectors. To address these discrepancies, we leverage recent advances in embedding space analysis and graph projection techniques, enabling the joint projection of transformations from diverse encoder architectures (Duderstadt et al., 2023). This encoding-centric analysis facilitates direct comparison of entire embedding spaces generated by different models.

Such an approach is especially valuable when downselecting from a large pool of candidate encoders. By quantifying representational similarity, we avoid exhaustive testing: models with highly similar encoding geometries are likely to yield comparable performance on a target task, meaning only representative models from each group need to be evaluated. Likewise, this methodology aids in constructing a taxonomy of applicable models tailored to specific tasks or domains.

Within a joint projection, each data point is represented N times, where N is the number of models under consideration. To assess the significance of differences between two representations of the same datum, we model the null distribution of representation distances, drawing on theoretical foundations from random graph literature. For a single model, resampling (via perturbation and bootstrapping) yields a rejection radius: a threshold delineating typical within-model embedding variation. When the paired representation of a sample from a different model falls outside of this radius, it indicates a materially different encoding of that sample. The probability of rejection tends to correlate with discrepancies in encoder training data and significant representational differences.

Finally, this technique can be iteratively applied: by generating a distance matrix that quantifies distances between N projections of identical samples across all embedding sets, and then applying dimensionality reduction again on the distance matrix, we obtain a streamlined, task-relevant comparison of model representations via a single point representative of each embedding set / model representation.

Like performance metrics for downstream tasks, each of these methodologies provides a piece of the full picture, and collectively they can provide a more comprehensive view of a model's representational skill and potential generalizability.


## Section B: The Framework

---
---

[Link to original repository](https://github.com/VMarsocci/pangaea-bench)

We apply the Pangaea Bench benchmarking framework originally developed by the ESA Phi Lab team (Marsocci et al., 2024). While, there are other frameworks in development by other teams, Pangaea conveniently has many of the more popular remote sensing foundation models integrated into their pure pytorch framework along with a number of well established benchmarking datasets. Figure 11 shows the design of the Pangaea framework. It also includes a relatively simple dataset integration scheme along with additional decoder heads and encoder evaluation methodologies integrated by the team at Spatial Informatics Group. The limited dependency list is especially beneficial for small scale demonstrations such as this learning notebook. You will find similar themes found in previous chapters of the [Applied Deep Learning Book](https://github.com/NASA-EarthRISE/EarthRISE-Applied-Artificial-Intelligence-and-Deep-Learning-Book) such as dataloaders, pytorch modules, loss functions, etc. These are not exclusive to training end-to-end models and in fact, the workflow is almost exactly the same! The EOFMs are intended to solve the same segmentation, classification, or regression problems in a different manner. Here, we focus on the additional considerations that come with using an EOFM for segmentation.

<img style="display: block;
           margin-left: auto;
           margin-right: auto;
           width: 70%;"
           src="https://raw.githubusercontent.com/sig-gis/eofm-book/main/assets/pangaea_workflow.png"/>

<div style="text-align: center;">Figure 11. Diagram of Pangaea's general structure.</div>
<br/>

---

### Software/Hardware considerations

CUDA is the underlying software running NVidia GPUs. While it is not necessary, we recommend running this notebook in a CUDA enabled linux environment for convenience and setup simplicity. We also recommend installing python>=3.10 either standalone or through a conda managed environment. For running the model, we can estimate a VRAM required for storing the model using a simple calculation of # of parameters and precision. For example, a 300M parameter model at 32bit precision would roughly require 3.6GB to store the weights and optimizer states. Another 1.2 GB is required for the gradients of each of those parameters and a few more for the data itself. Likewise, the decoder used to extract from the internal representation also imposes VRAM / compute requirements. There's no universal formula here, but tools like PyTorch hooks, torch.cuda.memory_summary(), or profilers can help estimate how much compute you need. 12GB of VRAM is a good place to start and luckily there are free google colab options at this compute scale, but, there is a trend towards larger and more complex models which inevitably consume more compute.

#### Installing framework and requirements

We are installing directly from a cloned repos along with the available requirements.txt file. This may take a few moments as PyTorch is a fairly large package. Please install python and pip on your own and setup on your notebook environment that best suits your preferences.

In [ ]:
!git clone --branch v0.1  --single-branch https://github.com/sig-gis/pangaea-bench.git

!pip install -e "pangaea-bench"
!pip install -r pangaea-bench/requirements.txt

!wget -O - "https://huggingface.co/datasets/ibm-nasa-geospatial/hls_burn_scars/resolve/main/hls_burn_scars.tar.gz?download=true" | tar -xz -C data/hls_burn_scars

!git clone --branch v0.1  --single-branch https://github.com/sig-gis/calculate-flops.pytorch.git

#Replate <absolute_path_to> with the absolute path to the calflops repository you just cloned
!export PYTHONPATH=${PYTHONPATH}:<absolute_path_to>/calculate-flops.pytorch/

#### The 'torchrun' Command

The torchrun command is a utility provided by PyTorch to launch distributed training jobs across multiple processes, nodes, or GPUs. It replaces the older torch.distributed.launch and is part of the torch.distributed module. torchrun is designed to be simple and flexible, making it easier to scale up training scripts with minimal changes. At its core, torchrun sets up the environment variables necessary for distributed training, such as RANK, WORLD_SIZE, and MASTER_ADDR, and then spawns multiple processes, one per GPU or per node, depending on the configuration. This enables parallel training using frameworks like DistributedDataParallel (DDP), which helps synchronize gradients and reduce training time significantly. This will be especially useful for training larger networks however, for the purposes of this demonstration we will only use it to run on a single machine

The Pangaea benchmarking framework is a lightweight wrapper around this command to handle most of boilerplate code that comes with training a neural network. This includes establishing the training loop, calculating metrics, logging, as well as the datasets/dataloaders associated with the benchmarks. [See torchrun documentation on environment variables for more details](https://docs.pytorch.org/docs/stable/elastic/run.html). There are several command options available which can be found in ./pangaea-bench/configs. We only use a subset below in this demonstration notebook but there are many more available. Config files associated with the dataset, decoder, and encoder are associated with a PyTorch module that is instantiated by the framework based on the configurations provided.

```bash
!torchrun pangaea-bench/pangaea/run.py \                                ##### The torchrun entry command
    --config-name=train \                                               ##### configuration name which can be found in ./pangaea-bench/configs
    work_dir=checkpoints \                                              ##### the directory relative to the working directory to store checkpoint outputs
    dataset=mtbs \                                              ##### the dataset config file found in ./pangaea-bench/configs
    encoder=dofa\                                                       ##### the encoder config file found in ./pangaea-bench/configs
    decoder=seg_fcn\                                                    ##### the decoder config file found in ./pangaea-bench/configs
    preprocessing=seg_default\                                          ##### preprocssing steps associated with the task type (e.g normalizing images)
    criterion=cross_entropy \                                           ##### the loss function on which to optimize the training cycle
    task=segmentation \                                                 ##### task type (others in clude classification and regression)
    use_wandb=true \                                                    ##### whether to use the online logging software
    task.trainer.n_epochs=16 \                                          ##### number of epochs to limit training
    task.trainer.log_interval=1 \                                       ##### logging interval of calculated metrics
    task.trainer.ckpt_interval=16 \                                     ##### how often to save checkpoints based on metrics
    encoder.encoder_weights=pretrained_models/DOFA_ViT_base_e100.pth    ##### path to pretarined weights of the FM

#### Logging and Model Examination

Weights & Biases (WandB) is a popular tool for experiment tracking, model monitoring, and collaboration in machine learning workflows. It integrates seamlessly with PyTorch, TensorFlow, Keras, and other frameworks, allowing users to log training metrics, visualize results in real-time, and compare model performance across runs. Pangaea benchmark happens to have a pre-baked integration of wandb which is easily accessed using WandB's api key authentication protocol. [See WandB documentation on environment variables for more details](https://docs.wandb.ai/guides/track/environment-variables/)

## Section C: Encoder Assessments

### Frozen vs Unfrozen vs Random vs LoRA

Testing frozen, unfrozen, and randomly initialized pretrained encoders is useful in understanding the value and applicability of transfer learning for a specific task. A frozen encoder uses pretrained weights without updating them during training (Marsocci et al., 2024). This setup helps evaluate how useful the pretrained features are on their own, especially in cases where the target dataset is small or the model is prone to overfitting. In contrast, an unfrozen encoder allows those pretrained weights to be updated, enabling the model to adapt its learned representations to better suit the target task. This approach often yields better performance when sufficient labeled data is available and the task deviates from the original pretraining domain however, requires that additional gradients be computed and stored. On the other hand, a randomly initialized encoder serves as a baseline, providing a measure of how well the model performs without any prior knowledge. Comparing results from this setup against those using pretrained weights helps quantify the benefits of pretraining in terms of training time and overall final performance. It's worth noting that depending on the task and the amount of available training data, the randomly initialized encoder may never achieve the same results as an unfrozen pretrained model. This behavior is still an ongoing area of research.

Overall, these three basic configurations allow for a comprehensive assessment of whether transfer learning is useful, whether fine-tuning improves results, and whether pretraining is necessary at all for a specific task. There is a fourth method which we will not use here called Low Rank Adaption (LoRA) and is somewhat of a compromise between a fully frozen and unfrozen encoder (Hu et al., 2021). LoRA is a technique used to efficiently fine-tune large pretrained neural networks by injecting small, trainable weight matrices into the model, while keeping the original weights frozen. This is typically reserved for larger models (i.e greater than 300m parameters). Traditional fine-tuning updates all parameters of a model, which becomes expensive for large models. LoRA avoids this by approximating the weight updates as the product of two low-rank matrices, significantly reducing the number of trainable parameters. This low-rank decomposition acts as a bottleneck, enforcing efficiency and reducing overfitting. LoRA has gained popularity particularly with large language models. It also enables modular fine-tuning where separate LoRA modules can be swapped in for different tasks making it attractive for applications requiring task specialization without retraining the entire model.


### Highlighted Models
For the encoder evaluation, we will take a look at a large set of pretrained encoders available within Pangaea-Bench. From there, we will focus fine-tuning on a single task for 3 frozen encoder variations:  Dynamic One-For-All (DOFA), Contrastive Radar-Optical Masked Autoencoder (CROMA), and Prithvi 1.0 (described below). Below is a brief introduction to the models we will focus on in the fine-tuning process and beyond.
#### DOFA

DOFA is a transformer-based model (Figure 12a) that is optimized on both a mask image modeling objective and self distillation objective (Figure 12b). DOFA's major contribution is their wavelength-conditioned dynamic patch embedding in which a the central wavelength of a given sensor is used to derive the weights for processing the respective data (Xiong et al., 2024). It is apparent from this that the model developers' approach is targeting flexibility and generalizability across multiple modalities and does so fairly well. And yet, there are still open questions. For example, wavelength and reflectance does not necessarily translate to amplitude data from SAR, does this matter? Benchmarking metrics will tell you no but do keep in mind the underlying reasons for these behaviors.

<img style="display: block;
           margin-left: auto;
           margin-right: auto;
           width: 70%;"
           src="https://raw.githubusercontent.com/sig-gis/eofm-book/main/assets/dofa.png"/>

<div style="text-align: center;">Figure 12: (a) Architecture design. DOFA builds on masked image modeling by processing input images with any number of channels within a single framework. (b) Dynamic weight generator and continual training framework from (Xiong et al., 2024). </div>

#### Prithvi 1.0

Prithvi v1.0 is one of the earliest foundation models and the simplest of the three models tested in this demonstration (Jakubik et al., 2023). Developed by the NASA IMPACT team, Prithvi 1.0 (Figure 13) is a masked autoencoder trained exclusively on Harmonized Landsat and Sentinel (HLS) data, with an added module for handling spatial and temporal context. The Prithvi model's training on HLS data provides an interesting opportunity for a discussion in our benchmarking: do we expect a model pretrained on HLS data to perform the best on tasks utilizing similar HLS imagery? Equally, to what extent do we expect models to adapt to similar but distinct modalities, especially those which overlap such as Sentinel-2 and non-harmonized Landsat.


<img style="display: block;
           margin-left: auto;
           margin-right: auto;
           width: 70%;"
           src="https://raw.githubusercontent.com/sig-gis/eofm-book/main/assets/prithvi.png"/>

<div style="text-align: center;">Figure 13: Masked autoencoder diagram for Prithvi's pretraining from (Jakubik et al., 2023).</div>

#### CROMA

CROMA (Figure 14) is another transformer-based model whose pretraining approach combines both MIM and contrastive learning (Fuller et al., 2023). Rather than contrasting positive and negative samples, the CROMA framework contrasts optical and radar data using separate encoders then combines those embeddings using cross attention with an additional transformer encoder. The output of this joint encoder is then randomly masked and trained in a typical encoder-decoder MIM style. The optical encoder requires all 12 spectral bands from Sentinel 2 while the radar encoder requires the 2 polarization bands from Sentinel 1. In our downstream task, we are limited to 6 harmonized spectral bands. How can we expect this to impact our results?

<img style="display: block;
           margin-left: auto;
           margin-right: auto;
           width: 70%;"
           src="https://raw.githubusercontent.com/sig-gis/eofm-book/main/assets/chroma.png"/>

<div style="text-align: center;">Figure 14: (Left) Pretraining framework for CROMA. (Right) Encoding workflow for leveraging internal representations.</div>

#### Encoder Weight-Matrix Analysis

Before we look at the encoders relative to the input data, let's evaluate the weight matrices. For this analysis, the dataset information is only used to extract metadata information within the pangaea library, so the results will be the same for hlsburnscars and mtbs configuration input values. It is important to note that some of these models support multi-modal inputs, or inputs from different instrument types (i.e. optical, radar, etc.). When additional modalities are brought in, some models leverage additional layers or encoders altogether. Here, both test cases are using optical only, so it is not a concern.

Below is the example command for CROMA. The commands for all other encoders can be found here: [Multi-Encoder Per-Layer Weight Matrix Analysis](https://github.com/sig-gis/pangaea-bench/blob/main/scripts/encoder_analysis/ww_mtbs.sh)

In [3]:
%%time
!pwd
%pushd /content/pangaea-bench/pangaea/encoder_analysis

!python3 weight_watcher.py dataset=mtbs encoder=croma_optical \
    preprocessing=seg_resize_input_layer task=segmentation

#Move output to location for longer-term storage
!mv img/* ww/croma/
!mv ww-img/* ww/croma/

!python3 weight_watcher.py dataset=mtbs encoder=dofa \
    preprocessing=seg_resize_input_layer task=segmentation

#Move output to location for longer-term storage
!mv img/* ww/dofa/
!mv ww-img/* ww/dofa/

!python3 weight_watcher.py dataset=mtbs encoder=prithvi \
    preprocessing=seg_resize_input_layer task=segmentation

#Move output to location for longer-term storage
!mv img/* ww/prithvi/
!mv ww-img/* ww/prithvi/


/content/pangaea-bench/pangaea/encoder_analysis
/content/pangaea-bench/pangaea/encoder_analysis
INFO - 06/06/26 20:32:24 - 0:00:00 - ============ Initialized logger ============
INFO - 06/06/26 20:32:24 - 0:00:00 - 'batch_size': 1,
                                      'ckpt_dir': '???',
                                      'data_replicate': 1,
                                      'dataset': {'_target_': 'pangaea.datasets.mtbs.MTBSBurnSeverity',
                                                  'auto_download': False,
                                                  'bands': {'optical': ['B2', 'B3', 'B4', 'B8A', 'B11', 'B12']},
                                                  'classes': ['Background', 'Unburned to Low', 'Low', 'Moderate',
                                                              'High', 'Increased Greenness', 'Non-Mapping Area'],
                                                  'data_max': {'optical': [1.35165, 1.38805, 1.48845, 1.5638, 1.7758,
               

There are additional details produced within plots generated by the commands run, but we will use the per-layer $\alpha$ plots in Figure 15 to provide a top-level summary of the findings. These plots summarize each layer's empirical spectral density using the power-law exponent $\alpha$; Lower $\alpha$ values indicates heavier tails and potential overfitting, whereas higher $\alpha$ values suggest lighter tails and possible underfitting

CROMA appears to have the most consistently well-trained layers, with most of the $\alpha$ values staying within the desired [2,5] range, and the deviation outside of this range being very minimal. Prithvi appears to have a handful of layers that are underfit to a small degree, and DOFA has a somewhat evenly distributed set of layers throughout the model that appear to be overfit.

There are a few layers that deviate past 5 at the beginning of each model, which is a pattern identified in previous studies of vision models using the weight watcher tool. The current hypothesis is that most high-level features learned in early layers of vision models are similar across all scenes in a large training set, so the early layers in a model, typically the ones that learn these high-level features tend towards overfitting / a lack of generalizability. Here, the deviations only appear to be minor.

<img style="display: block;
           margin-left: auto;
           margin-right: auto;
           width: 50%;"
           src="https://raw.githubusercontent.com/sig-gis/eofm-book/main/assets/Combined_Alpha_vs_Depth_Plots.png"/>

<div style="text-align: center;">Figure 15. Plots containing the $\alpha$ summary parameter from the model weight matrix analysis done for CROMA (top), Prithvi (center), and DOFA (bottom).</div>
</br>


This analysis not only informs us about performance considerations when using these models with frozen weights, but also can provide some insight into what we need to take into account if we are to fine-tune all or a subset of the weights of these encoders. For example:

**1)** Which models' overfit layers might be more prone to a phenomenon called catastrophic forgetting, where performance significantly dips after fine-tuning due to the information gained in pretraining being lost or overwritten.

**2)** Which model's layers may need more time to learn / optimize during the fine-tuning phase, due to being underfit in pretraining.


#### Resource Requirements - FLOPS, MACS, and Parameter Counts

Next, we will take a look at the amount of computational and storage requirements needed for each model. Spectral analysis tells us something about how encoders use their parameters; we also need to know how many parameters, and how much compute, they require in the first place.

Below is the example command for CROMA. The commands for all other encoders can be found here: [MTBS Multi-Encoder Resource Consumption Estimation](https://github.com/sig-gis/pangaea-bench/blob/main/scripts/encoder_analysis/compute_flops_mtbs.sh). Table 1 provides the results of the computation for a larger set of models supported by Pangaea. Here we use a small amount of data to pass into the resource utilization profiler, but relative resource usage does not change across the two datasets in being used here, and there are only minor shifts in the end results. Therefore, we only look at output generated using the MTBS data here. In subsequent steps we will look at both separately. As with the weight-matrix analysis,  if other modalities are being used, the resource consumption outputs will likely change.


In [16]:
%%time

import os
os.environ['PYTHONPATH'] += ":/content/calculate-flops.pytorch/"
#!python3 compute_flops.py dataset=hlsburnscars   encoder=croma_optical  preprocessing=seg_irregular_images    task=segmentation
#!python3 compute_flops.py dataset=hlsburnscars   encoder=dofa  preprocessing=seg_irregular_images    task=segmentation
!python3 compute_flops.py dataset=hlsburnscars   encoder=prithvi  preprocessing=seg_irregular_images    task=segmentation

INFO - 06/07/26 17:13:27 - 0:00:00 - ============ Initialized logger ============
INFO - 06/07/26 17:13:27 - 0:00:00 - 'batch_size': 1,
                                      'ckpt_dir': '???',
                                      'data_replicate': 1,
                                      'dataset': {'_target_': 'pangaea.datasets.hlsburnscars.HLSBurnScars',
                                                  'auto_download': True,
                                                  'bands': {'optical': ['B2', 'B3', 'B4', 'B8A', 'B11', 'B12']},
                                                  'classes': ['Not burned', 'Burn scar'],
                                                  'data_max': {'optical': [0.0, 0.0, 0.0, 0.0, 0.0, 0.0]},
                                                  'data_mean': {'optical': [0.033349706741586264,
                                                                            0.05701185520536176, 0.05889748132001316,
                                         


<img src="https://raw.githubusercontent.com/sig-gis/eofm-book/main/assets/Resource_Consumption_Table.png"/>

<div style="text-align: center;">Table 1. Resource consumption of the three encoders relative to common pretrained models. Note the diversity in amount of calculations. We will touch on this in the following section.</div>
</br>

Here we can see that CROMA, DOFA, and Prithvi stand out as some of the largest models with number of operations for forwards and backwards passes also in some of the highest counts. These models are being used here to provide an example and do not necessarily outperform all others when compared to every other permutation possible within Pangaea **and those without**. The authors re-emphasize that the smallest and most optimized (least resource intensive) model that provides a good representation of the features of interest in the dataset and a good enough (typically task dependent) performance on the ultimate task should be chosen.


#### Embedding generation

Next will take the input dataset and pass it through the pretrained encoders to get model-specific representations, or embeddings for each input scene. Below are example commands for the CROMA encoder and both the MTBS and HLS Burn Scars datasets. We also generate these for 16 other encoders for both cases (pretrained and randomly initialized where available). Additional examples can be found here: [MTBS Multi-Encoder Embedding Generation Script](https://github.com/sig-gis/pangaea-bench/blob/main/scripts/encoder_analysis/embed_gen_mtbs.sh) and [HLS Multi-Encoder Embedding Generation Script](https://github.com/sig-gis/pangaea-bench/blob/main/scripts/encoder_analysis/embed_gen_hls_burnscars.sh)

In [ ]:
%%time

!mkdir ./embeddings/

!python3 embed.py dataset=hlsburnscars encoder=croma_optical preprocessing=seg_irregular_images task=segmentation
#!python3 embed.py dataset=mtbs encoder=croma_optical preprocessing=seg_default task=segmentation

!python3 embed.py dataset=hlsburnscars encoder=dofa preprocessing=seg_irregular_images task=segmentation
#!python3 embed.py dataset=mtbs encoder=dofa preprocessing=seg_default task=segmentation

!python3 embed.py dataset=hlsburnscars encoder=prithvi preprocessing=seg_irregular_images task=segmentation
#!python3 embed.py dataset=mtbs encoder=prithvi preprocessing=seg_default task=segmentation

mkdir: cannot create directory ‘./embeddings/’: File exists
INFO - 06/08/26 05:52:34 - 0:00:00 - ============ Initialized logger ============
INFO - 06/08/26 05:52:34 - 0:00:00 - 'batch_size': 1,
                                      'ckpt_dir': '???',
                                      'data_replicate': 1,
                                      'dataset': {'_target_': 'pangaea.datasets.hlsburnscars.HLSBurnScars',
                                                  'auto_download': True,
                                                  'bands': {'optical': ['B2', 'B3', 'B4', 'B8A', 'B11', 'B12']},
                                                  'classes': ['Not burned', 'Burn scar'],
                                                  'data_max': {'optical': [0.0, 0.0, 0.0, 0.0, 0.0, 0.0]},
                                                  'data_mean': {'optical': [0.033349706741586264,
                                                                            0.05701185520536176, 0.

#### UMAP Projection and KNN Graph Generation

Now that we have the embeddings we can project them onto a lower-dimension manifold and generate a KNN-Graph from that represenation. Below are example commands for CROMA. The commands for all other encoders can be found here: [MTBS Multi-Encoder UMAP Projection and KNN-Graph Generation](https://github.com/sig-gis/pangaea-bench/blob/main/scripts/encoder_analysis/umap_and_knn_graph_gen_mtbs.sh) and [HLS Burn Scars Multi-Encoder UMAP Projection and KNN-Graph Generation](https://github.com/sig-gis/pangaea-bench/blob/main/scripts/encoder_analysis/umap_and_knn_graph_gen_hls.sh)

In [ ]:
%%time
!python3 knn_graph_gen.py dataset=hlsburnscars \
                          encoder=croma_optical \
                          preprocessing=seg_irregular_images \
                          criterion=cross_entropy \
                          task=segmentation

#!python3 knn_graph_gen.py dataset=mtbs \
#                          encoder=croma_optical \
#                          preprocessing=seg_irregular_images \
#                          criterion=cross_entropy \
#                          task=segmentation

!python3 knn_graph_gen.py dataset=hlsburnscars \
                          encoder=dofa \
                          preprocessing=seg_irregular_images \
                          criterion=cross_entropy \
                          task=segmentation

#!python3 knn_graph_gen.py dataset=mtbs \
#                          encoder=croma_optical \
#                          preprocessing=seg_irregular_images \
#                          criterion=cross_entropy \
#                          task=segmentation

!python3 knn_graph_gen.py dataset=hlsburnscars \
                          encoder=prithvi \
                          preprocessing=seg_irregular_images \
                          criterion=cross_entropy \
                          task=segmentation

#!python3 knn_graph_gen.py dataset=mtbs \
#                          encoder=croma_optical \
#                          preprocessing=seg_irregular_images \
#                          criterion=cross_entropy \
#                          task=segmentation

We can use these plots to first do a qualitative analysis. If we compare these UMAP plots to those in Figures 16 and 17, where we can see weak to moderate separations of the classes for all models and can take away the fact that these are not an easy task for these models out of the box.

##### UMAP Analysis HLS Burn Scars

<img style="display: block;
           margin-left: auto;
           margin-right: auto;
           width: 50%;"
           src="https://raw.githubusercontent.com/sig-gis/eofm-book/main/assets/HLS_UMAP.png"/>
<div style="text-align: center;">Figure 16. A plot of the embeddings generated from CROMA (left), Dofa (center), and Prithvi (left) for samples from the test split of the HLS Burn Scars dataset projected onto a low-dimension (2D) manifold via UMAP and overlayed with their associated target value. The key for the targets on the bottom right.</div>
</br>

In all three cases, to varying degrees, there is an intermixing of the label values. CROMA appears to do the best at separating out subsets of the classes in different ways, and Prithvi does keep some distinction between them, while DOFA appears to have the hardest time representing the classes distincly in its embedding space, as demonstrated by a lack of clear separation of the classes anywhere in the plot.


##### UMAP Analysis MTBS

<img style="display: block;
           margin-left: auto;
           margin-right: auto;
           width: 50%;"
           src="https://raw.githubusercontent.com/sig-gis/eofm-book/main/assets/MTBS_UMAP.png"/>

<div style="text-align: center;">Figure 17. A plot of the embeddings generated from CROMA (left), DOFA (center), and Prithvi (left) for samples from the test split of the MTBS dataset projected onto a low-dimension (2D) manifold via UMAP and overlayed with their associated target value. The key for the targets on the bottom right.</div>
</br>

Again, but at a higher rate than in the HLS Burn Scars plots, there is an intermixing of the label values.  Both Prithvi and CROMA appear to do a better job of separating out subsets of the classes in different ways, while DOFA appears to have the hardest time representing the classes distincly in its embedding space, as demonstrated by a lack of clear separation of the classes anywhere in the plot.


As mentioned before, this methodology is not perfect, and we are removing representative information from embedded samples by reducing the feature space. However, trade-offs have to be made for interpretability and analysis of model performance because results need to be human-interpretable in order for them to be useful. If a model is more easily representing distinctions between samples in a simplified way at a higher dimension, those distinctions will likely be picked up in this lower dimension space.

#### Inter-Model Embedding Comparisons Using Data Kernels

Here, we take the KNN-graphs previously generated and use a data-kernel-based projection to project all embedding samples to a uniform feature space and do subsequent analysis. Below is the example command for CROMA. The commands for all other encoders can be found here: [MTBS Multi-Encoder Embedding Comparison](https://github.com/sig-gis/pangaea-bench/blob/main/scripts/encoder_analysis/embed_compare_mtbs.sh) and [HLS Burn Scars Multi-Encoder Embedding Comparison](https://github.com/sig-gis/pangaea-bench/blob/main/scripts/encoder_analysis/embed_compare_hls.sh)

In [ ]:
!python3 data_kernel_analysis.py dataset=hlsburnscars   task=segmentation

#!python3 data_kernel_analysis.py dataset=mtbs   task=segmentation

INFO - 06/07/26 21:09:36 - 0:00:00 - ============ Initialized logger ============
INFO - 06/07/26 21:09:36 - 0:00:00 - 'batch_size': 1,
                                      'ckpt_dir': '???',
                                      'data_replicate': 1,
                                      'dataset': {'_target_': 'pangaea.datasets.hlsburnscars.HLSBurnScars',
                                                  'auto_download': True,
                                                  'bands': {'optical': ['B2', 'B3', 'B4', 'B8A', 'B11', 'B12']},
                                                  'classes': ['Not burned', 'Burn scar'],
                                                  'data_max': {'optical': [0.0, 0.0, 0.0, 0.0, 0.0, 0.0]},
                                                  'data_mean': {'optical': [0.033349706741586264,
                                                                            0.05701185520536176, 0.05889748132001316,
                                         

For both datasets, we start by generating a scatter plot of all samples to visualize similarities and differences (Figure 18 and 22). Next we reduce the dimensionality again, creating a manifold where each set of embeddings is represented by a single point (Figure 19 and 23). With joint projections from paired models, we can model the null distribution of differences between the two representations using ideas drawn random graph theory to define this baseline. In other words, paired data that falls outside of a rejection radius is seen to be represented significantly different.

Statistical tests between these representations provide insight into knowledge gaps which are unique to each model, highlighting differences in what they have actually learned or failed to capture either as a result of their training regime or their data selection process. These tests suggest that, although foundation models are individually designed to serve as unifying generalists tools, the representations they learn remain distinct from one another.

##### Embedding Space Comparisons - HLS Burn Scars

Figure 18 provides sample-wise scatter plots for a subset of samples projected into the uniform collective embedding space. This is shown for the models in focus (top left), a larger set of models supported by Pangaea-Bench (top right), as well as individual plots for each model in focus, with an additional visualization of spatial density (bottom) which will come in handy in this analysis.

<img style="display: block;
           margin-left: auto;
           margin-right: auto;
           width: 50%;"
           src="https://raw.githubusercontent.com/sig-gis/eofm-book/main/assets/Embed_Scatter_HLS.png"/>

<div style="text-align: center;">Figure 18. A depiction of the joint embedding projection for the embeddings of a subset of the HLS Burn Scar data for the three models of focus here (top left) as well as a larger set of models supported by Pangaea-Bench (top right). The bottom row depicts a zoom in of CROMA's (left), Dofa's (center), and Prithvi's (right) samples in the embedding space, with opacity being used here as a representation of spatial density of samples. The red box in the top right figure indicates the subsetted area in the full embedding where these model's have samples projected. The plots on the top left and bottom row contain only this subset area. </div>
</br>

In Figure 19 we can see that CROMA, DOFA, and Prithvi are relatively close together, although it does not appear that they belong to a distinct single cluster of models.

<img style="display: block;
           margin-left: auto;
           margin-right: auto;
           width: 45%;"
           src="https://raw.githubusercontent.com/sig-gis/eofm-book/main/assets/Embed_Space_Dist_Mtx_HLS.png"/>

<div style="text-align: center;">Figure 19. A depiction of collapsing the embedding spaces into single points to visualize and measure population-level representational differences. </div>
</br>

Figure 20 and 21 shows pairwise comparison plots the pairwise null distributions of differences between the two representations of the same samples, as described above.  We can see that the representations are identified as somewhat similar, but clearly not the same. This is demonstrated by the number of samples with small p-values (less opaque samples in Figure 20) as well as null-distribution plots in Figure 21, demonstrating slight shifts towards a uniform distribution (blue lines in Figure 21).


<img style="display: block;
           margin-left: auto;
           margin-right: auto;
           width: 70%;"
           src="https://raw.githubusercontent.com/sig-gis/eofm-book/main/assets/Scatter_P_Val_HLS.png"/>

<div style="text-align: center;">Figure 20. Scatterplots of the embedding projections being used to measure population-level representational shifts between pairs of models. Here, the sample set from the model being compared against is plotted and the model we are measuring deviation from has its samples overlayed on top. The opacity has an inverse relationship to the p-value of a given sample. All values with a p-value close to or equal to 1 are colored white here and therefore do not show up. </div>
</br>

<img style="display: block;
           margin-left: auto;
           margin-right: auto;
           width: 70%;"
           src="https://raw.githubusercontent.com/sig-gis/eofm-book/main/assets/P_Value_Dist_HLS.png"/>
<div style="text-align: center;">Figure 21. The distribution of p-values under the null hypothesis for each ordered pair of models.</div>
</br>

The comparisons of Prithvi and CROMA samples (center top and bottom) as well as Prithvi and DOFA (right top and bottom) provide an interesting aside. Notice that when CROMA or DOFA embeddings are held as the reference set and Prithvi as the comparison set we get the a closer measure of similarity in the set of plots, while we see a complete rejection of the null hypothesis when the roles of the datasets are swapped. This is due to the difference in spread between each model's samples in the collective embedding space. The rejection radius for CROMA and DOFA are larger than for Prithvi. Thus, when we use CROMA or DOFA as reference, more sample tests accept the null hypothesis. We can see these patterns show up in the density-scaled scatter plots in Figure 18 as well as the inverse-p-value scaled ones in Figure 20. All of these analyses are consistent with the varying qualities observed in Figures 16 & 17.

##### Embedding Space Comparisons - MTBS

Figure 22 provides sample-wise scatter plots for a subset of samples projected into the uniform collective embedding space for the MTBS case. Because the re-projected embeddings for these models and this data span a wider set of the embedding field, we do not subsample for the 3-model and single model scatterplots, like in Figure 18.

<img style="display: block;
           margin-left: auto;
           margin-right: auto;
           width: 70%;"
           src="https://raw.githubusercontent.com/sig-gis/eofm-book/main/assets/Embed_Scatter_MTBS.png"/>

<div style="text-align: center;">Figure 22. A depiction of the joint embedding projection for the embeddings of the MTBS dataset and the three models of focus here (top left) as well as a larger set of models supported by Pangaea-Bench (top right). The bottom row depicts CROMA's (left), Dofa's (center), and Prithvi's (right) samples in the embedding space, with opacity being used here as a representation of spatial density of samples.</div>
</br>

In the reduced dimension single-point mapping in Figure 23 we can see that CROMA, DOFA, and Prithvi are all in distinct clusters, meaning that this measure of their representational structure shows strong distinctions. This gives us one point of information in our model selection and representation analysis journey, but should not be used as a comprehensive indicator on its own.

<img style="display: block;
           margin-left: auto;
           margin-right: auto;
           width: 45%;"
           src="https://raw.githubusercontent.com/sig-gis/eofm-book/main/assets/Embed_Space_Dist_Mtx_MTBS.png"/>

<div style="text-align: center;">Figure 23. A depiction of collapsing the embedding spaces into single points to visualize and measure population-level representational differences for the MTBS dataset test case. </div>
</br>

Figure 24 and 25 show different visualizations from pairwise comparisons of the null distributions of differences between the two representations of the same samples, as described above. Here again, we can see that the representations are distinct. This is demonstrated by the number of samples with similarly high p-values as the burn scar case as well as null-distribution plots in Figure 21 demonstrating no shift towards a uniform distribution (blue lines in Figure 21). In this case, that no model's class separation really stands out in the UMAP plots and all models have distinct representations.


<img style="display: block;
           margin-left: auto;
           margin-right: auto;
           width: 70%;"
           src="https://raw.githubusercontent.com/sig-gis/eofm-book/main/assets/Scatter_P_Val_MTBS.png"/>

<div style="text-align: center;">Figure 24. Scatterplots of the MTBS embedding projections being used to measure population-level representational shifts between pairs of models. Here, the sample set from the model being compared against is plotted and the model we are measuring deviation from has its samples overlayed on top. The opacity has an inverse relationship to the p-value of a given sample. All values with a p-value close to or equal to 1 are colored white here and therefore do not show up. </div>
</br>

<img style="display: block;
           margin-left: auto;
           margin-right: auto;
           width: 70%;"
           src="https://raw.githubusercontent.com/sig-gis/eofm-book/main/assets/P_Value_Dist_MTBS.png"/>
<div style="text-align: center;">Figure 25. The distribution of p-values under the null hypothesis for each ordered pair of models and the MTBS data.</div>
</br>


The comparison of Prithvi and CROMA samples (center top and bottom) demonstrates a similar phenomenon as the ones described in the previous case. Again, when CROMA embeddings are held as the reference set and Prithvi as the comparison set we get the a closer measure of similarity , while we see a complete rejection of the null hypothesis when the roles of the datasets are swapped. This is due to the difference in spread between each model's samples in the collective embedding space. The rejection radius for CROMA is larger than for Prithvi, given the spread of a larger set of samples from CROMA. Thus, when we use CROMA as reference, more sample tests accept the null hypothesis. The difference is significantly less when looking at the plots for DOFA and Prithvi because the largest density of DOFA's embeddings is farther away from that of Prithvi's, when compared to CROMA's. The case is very similar when comparing DOFA and CROMA. We can see these patterns show up in the density-scaled scatter plots in Figure 17 as well as the inverse-p-value scaled ones in Figure 19. Again, this is consistent with the varying qualities observed in Figures 16 & 17.

### Variability and spread

Models produce distinct representations even when given the same input. This arises from differences in architecture, training data, and objectives, leading to varying interpretations that affect performance, generalizability, and trust in model outputs. This distinction is especially important when considering the encoder component, which is solely responsible for transforming raw input into that latent representation. Since the encoder defines what information is preserved, emphasized, or discarded, any differences in its learned mapping directly shape the structure and spread of the embedding space. For instance, two encoders may process the same image and yield representations that differ not just in detail, but in what aspects of the scene are even considered salient such as texture vs. shape, global vs. local features, or background vs. foreground elements.

With that said, greater variability and spread in the projected embedding space, as in the case of the CROMA model, does not necessarily translate to improved utility. A model with a more diffuse spread may have greater capacity in separating subtle differences between input types yet it may also be vulnerable to noise or irrelevant distinctions. This added burden can lead to increased complexity in downstream task design, reduced robustness, or even degraded performance if the embeddings fail to cluster relevantly similar inputs for the task at hand. In such cases, the downstream model must work harder to reimpose structure on a representation space that is less naturally organized, potentially offsetting the advantages of the encoder’s expressiveness.

These tools can not only be used to inter-compare distinct encoders, but can also be used to visualize what happens to a model's representations when processes are changed. For instance, they can be used to look at representational differences of a single encoder before and after fine-tuning or given two different pretraining dataset splits (Duderstadt et al., 2023).

Identifications of embedding level differences and large-scale representation gaps are crucial and can be quantitatively measured using these tools.  These methodologies to rigorously inter-compare representative capabilities are crucial for both better understanding encoder performance and for further architecture development.

## Section D: Example Fine Tuning

### Background

Now that we've established some of the major concepts underpinning the foundation models, we can move forward with fine-tuning the models. Along with the 3 FMs highlighted above, we will train the 5 different decoders described in section A for a total of 15 model variations. It is clear that, for comparative studies of foundation models, compute becomes a key bottleneck—especially when aiming for systematic comparisons across models tasks, or training regimes. For the sake of this demonstration, we limit each training run to a batch size of 8 for 16 epochs an on a single RTX 4000 workstation GPU with 12GB of VRAM. A machine of this size is readily available on any cloud compute platform.

We target two fire related tasks: segmenting burn scars in HLS scenes and multiclass segmentation of burn intensity. Both are HLS-based derived benchmarks, imagery source that is unique in that it sources its optical data from two satellite missions (Landsat and Sentinel) and in doing so, provides a relatively level testbed for models trained on either. Nevertheless, it’s important to acknowledge the nature of the benchmark datasets used. While it has been carefully curated to support robust model evaluation, it may not fully reflect the diversity or complexity of real-world fire scenarios with potentially drastically different landscape dynamics, data availability, and task scope. The dataset is limited in geographic scope, vegetation types, and imaging conditions, which means performance metrics obtained here may not generalize well to all fire contexts such as risk mapping. As such, results from this benchmark should be interpreted with consideration of these constraints.

### HLS BURN SCARS Training Commands


In [ ]:

## DOFA

!torchrun pangaea-bench/pangaea/run.py \
    --config-name=train \
    work_dir=checkpoints \
    dataset=hlsburnscars \
    encoder=dofa\
    decoder=seg_muster\
    preprocessing=seg_default\
    criterion=cross_entropy \
    task=segmentation \
    use_wandb=true \
    task.trainer.n_epochs=16 \
    task.trainer.log_interval=1 \
    task.trainer.ckpt_interval=16 \
    encoder.encoder_weights=pretrained_models/DOFA_ViT_base_e100.pth

!torchrun pangaea-bench/pangaea/run.py \
    --config-name=train \
    work_dir=checkpoints \
    dataset=hlsburnscars \
    encoder=dofa\
    decoder=seg_fcn\
    preprocessing=seg_default\
    criterion=cross_entropy \
    task=segmentation \
    use_wandb=true \
    task.trainer.n_epochs=16 \
    task.trainer.log_interval=1 \
    task.trainer.ckpt_interval=16 \
    encoder.encoder_weights=pretrained_models/DOFA_ViT_base_e100.pth

!torchrun pangaea-bench/pangaea/run.py \
    --config-name=train \
    work_dir=checkpoints \
    dataset=hlsburnscars \
    encoder=dofa\
    decoder=seg_linear\
    preprocessing=seg_default\
    criterion=cross_entropy \
    task=segmentation \
    use_wandb=true \
    task.trainer.n_epochs=16 \
    task.trainer.log_interval=1 \
    task.trainer.ckpt_interval=16 \
    encoder.encoder_weights=pretrained_models/DOFA_ViT_base_e100.pth

!torchrun pangaea-bench/pangaea/run.py \
    --config-name=train \
    work_dir=checkpoints \
    dataset=hlsburnscars \
    encoder=dofa\
    decoder=seg_segformer\
    preprocessing=seg_default \
    criterion=cross_entropy \
    task=segmentation \
    use_wandb=true \
    task.trainer.n_epochs=16 \
    task.trainer.log_interval=1 \
    task.trainer.ckpt_interval=16 \
    encoder.encoder_weights=pretrained_models/DOFA_ViT_base_e100.pth

!torchrun pangaea-bench/pangaea/run.py \
    --config-name=train \
    work_dir=checkpoints \
    dataset=hlsburnscars \
    encoder=dofa\
    decoder=seg_upernet\
    preprocessing=seg_default \
    criterion=cross_entropy \
    task=segmentation \
    use_wandb=true \
    task.trainer.n_epochs=16 \
    task.trainer.log_interval=1 \
    task.trainer.ckpt_interval=16 \
    encoder.encoder_weights=pretrained_models/DOFA_ViT_base_e100.pth

## PRITHVI

!torchrun pangaea-bench/pangaea/run.py \
    --config-name=train \
    work_dir=checkpoints \
    dataset=hlsburnscars \
    encoder=prithvi \
    decoder=seg_muster\
    preprocessing=seg_default \
    criterion=cross_entropy \
    task=segmentation \
    use_wandb=true \
    task.trainer.n_epochs=16 \
    task.trainer.log_interval=1 \
    task.trainer.ckpt_interval=16 \
    encoder.encoder_weights=pretrained_models/Prithvi_EO_V1_100M.pt

!torchrun pangaea-bench/pangaea/run.py \
    --config-name=train \
    work_dir=checkpoints \
    dataset=hlsburnscars \
    encoder=prithvi \
    decoder=seg_fcn\
    preprocessing=seg_default \
    criterion=cross_entropy \
    task=segmentation \
    use_wandb=true \
    task.trainer.n_epochs=16 \
    task.trainer.log_interval=1 \
    task.trainer.ckpt_interval=16 \
    encoder.encoder_weights=pretrained_models/Prithvi_EO_V1_100M.pt

!torchrun pangaea-bench/pangaea/run.py \
    --config-name=train \
    work_dir=checkpoints \
    dataset=hlsburnscars \
    encoder=prithvi \
    decoder=seg_linear\
    preprocessing=seg_default \
    criterion=cross_entropy \
    task=segmentation \
    use_wandb=true \
    task.trainer.n_epochs=16 \
    task.trainer.log_interval=1 \
    task.trainer.ckpt_interval=16 \
    encoder.encoder_weights=pretrained_models/Prithvi_EO_V1_100M.pt

!torchrun pangaea-bench/pangaea/run.py \
    --config-name=train \
    work_dir=checkpoints \
    dataset=hlsburnscars \
    encoder=prithvi\
    decoder=seg_segformer\
    preprocessing=seg_default \
    criterion=cross_entropy \
    task=segmentation \
    use_wandb=true \
    task.trainer.n_epochs=16 \
    task.trainer.log_interval=1 \
    task.trainer.ckpt_interval=16 \
    encoder.encoder_weights=pretrained_models/Prithvi_EO_V1_100M.pt

!torchrun pangaea-bench/pangaea/run.py \
    --config-name=train \
    work_dir=checkpoints \
    dataset=mtbs \
    encoder=prithvi \
    decoder=seg_upernet\
    preprocessing=seg_default \
    criterion=cross_entropy \
    task=segmentation \
    use_wandb=true \
    task.trainer.n_epochs=16 \
    task.trainer.log_interval=1 \
    task.trainer.ckpt_interval=16 \
    encoder.encoder_weights=pretrained_models/Prithvi_EO_V1_100M.pt

## CROMA

!torchrun pangaea-bench/pangaea/run.py \
    --config-name=train \
    work_dir=checkpoints \
    dataset=hlsburnscars \
    encoder=croma_optical\
    decoder=seg_muster\
    preprocessing=seg_default \
    criterion=cross_entropy \
    task=segmentation \
    use_wandb=true \
    task.trainer.n_epochs=16 \
    task.trainer.log_interval=1 \
    task.trainer.ckpt_interval=16 \
    encoder.encoder_weights=pretrained_models/CROMA_large.pt

!torchrun pangaea-bench/pangaea/run.py \
    --config-name=train \
    work_dir=checkpoints \
    dataset=hlsburnscars \
    encoder=croma_optical\
    decoder=seg_fcn\
    preprocessing=seg_default \
    criterion=cross_entropy \
    task=segmentation \
    use_wandb=true \
    task.trainer.n_epochs=16 \
    task.trainer.log_interval=1 \
    task.trainer.ckpt_interval=16 \
    encoder.encoder_weights=pretrained_models/CROMA_large.pt

!torchrun pangaea-bench/pangaea/run.py \
    --config-name=train \
    work_dir=checkpoints \
    dataset=hlsburnscars \
    encoder=croma_optical\
    decoder=seg_linear\
    preprocessing=seg_default \
    criterion=cross_entropy \
    task=segmentation \
    use_wandb=true \
    task.trainer.n_epochs=16 \
    task.trainer.log_interval=1 \
    task.trainer.ckpt_interval=16 \
    encoder.encoder_weights=pretrained_models/CROMA_large.pt

!torchrun pangaea-bench/pangaea/run.py \
    --config-name=train \
    work_dir=checkpoints \
    dataset=hlsburnscars \
    encoder=croma_optical\
    decoder=seg_segformer\
    preprocessing=seg_default \
    criterion=cross_entropy \
    task=segmentation \
    use_wandb=true \
    task.trainer.n_epochs=16 \
    task.trainer.log_interval=1 \
    task.trainer.ckpt_interval=16 \
    encoder.encoder_weights=pretrained_models/CROMA_large.pt

!torchrun pangaea-bench/pangaea/run.py \
    --config-name=train \
    work_dir=checkpoints \
    dataset=hlsburnscars \
    encoder=croma_optical\
    decoder=seg_upernet\
    preprocessing=seg_default \
    criterion=cross_entropy \
    task=segmentation \
    use_wandb=true \
    task.trainer.n_epochs=16 \
    task.trainer.log_interval=1 \
    task.trainer.ckpt_interval=16 \
    encoder.encoder_weights=pretrained_models/CROMA_large.pt


### HLS BURN SCARS Results

Some things to consider:

Below, Figure 26 depicts the results from running the above commands aggregated into relatively simple charts. Keep in mind, we only train for 16 epochs for the purpose of a simple demonstration. This is a very small amount of time for a deep learning model. There is a reasonable chance that these metrics will change given enough training time, a different loss function, or a different set of hyperparameters. The framework allows for some level of configuration. On your own, try setting new configurations (see ./pangaea-bench/configs for details).

#### Training Graphs

In spite of a limited training window, we see a fairly distinct training curve which suggests the models have learned some useful features. Whether this is due to the pretraining or the simple nature of the task is still an open question. A straightforward test for that is to train using a randomly initialized encoder simply by adjusting the last configuration in each training command to null. In addition, there is a clear loss difference between each model with DOFA generally having higher cross entropy loss.

<img style="display: block;
           margin-left: auto;
           margin-right: auto;
           width: 70%;"
           src="https://raw.githubusercontent.com/sig-gis/eofm-book/main/assets/training_graph.png"/>
<div style="text-align: center;">Figure 26: Training loss per step for all model varations. Each model was optimized on cross entropy loss.</div>
</br>

#### Decoder Choice

The test metrics for the checkpoint saved at the final epoch, shown in in Figure 29, tell a slightly different story. Overall, the CROMA combinations appear to be the most effective in terms of overall performance metrics (F1 and IoU). The PRITHVI encoder consistently shows good performance across different decoders, with the Linear Probe surprisingly being the top performer in this group. The DOFA Linear Probe combination stands out as an outlier with extremely low performance for burn scar detection. The lower performance of the probe into DOFA suggests their internal representatioin may not be appropriate for this task and the decoder might be doing the bulk of the heavy lifting in the other runs, which aligns with our evaluation of the encoder analysis in the above sections. More experimentation is required to confirm this. It's worth noting that for even with the same encoder, varying the decoder has a visible effect on performance that is comparible in magnitude to the differences between encoders.

<img style="display: block;
           margin-left: auto;
           margin-right: auto;
           width: 70%;"
           src="https://raw.githubusercontent.com/sig-gis/eofm-book/main/assets/test_metrics.png"/>

<div style="text-align: center;">Figure 27: Test metrics per encoder-decoder variation</div>

#### Per Class Metrics

Looking at per class metrics in Figure 28 gives a bit more insight into what's happening. Aside from the DOFA-linear-probe outlier, the differences from model to model are realistically negligible, although further statistical testing would be recommended to confirm this. However, when we look at per class metrics, we see a more noticeable difference between each model's ability to segment the positive class. This suggests a class imbalance issue. While this may be an accurate representation of the task, this is something that needs to be taken into account when designing a benchmark dataset and evaluation framework.

<img style="display: block;
           margin-left: auto;
           margin-right: auto;
           width: 90%;"
           src="https://raw.githubusercontent.com/sig-gis/eofm-book/main/assets/per_class.png"/>

<div style="text-align: center;">Figure 28: Per class metrics</div>
</br>

#### Run Times and Compute Costs

Based on the above metrics alone, CROMA seems to be a top performer, nevertheless, we must consider one more aspect: compute. In Figure 29, we see that CROMA consumed up to ~3 times the memory and up to ~9 times the training time than some of the other models for the same number of epochs. This is consistent with the FLOP calculations found in Table 1. These are likely due to the large amount of dimensions in the CROMA model released compared to its peers (1024 as opposed to more common 768). In other words, for the same resources, DOFA combined with an even larger decoder trained for 128 epochs instead of 16 may very well outperform the CROMA models. In the same vein, Prithvi consumed a comparable amount of memory to CROMA but is far simpler and to implement and fine tunes several times faster. Theoretically, our models are modular and can be compared at a variety of sizes for each model. However, most models are released with pretrained versions only at a single size so comparisons must be taken for models as they are.

<img style="display: block;
           margin-left: auto;
           margin-right: auto;
           width: 60%;"
           src="https://raw.githubusercontent.com/sig-gis/eofm-book/main/assets/memory.png"/>
           
<img style="display: block;
           margin-left: auto;
           margin-right: auto;
           width: 60%;"
           src="https://raw.githubusercontent.com/sig-gis/eofm-book/main/assets/training_time.png"/>

<div style="text-align: center;">Figure 29: (Top) GPU VRAM usage per encoder-decoder combination. (Bottom) Run time in seconds to reach 16 epochs.</div>
</br>

#### Output Masks

We have all these metrics and they all tell a similar story. The first thing you should notice in from Figure 30 is that the annotation is not perfect. So the metrics above should be interpreted carefully. For example, the prithvi masks from all decoders look vastly different while the metrics say otherwise. When the benchmark dataset was created, chip labels with multiple annotations in close proximity were likely left unfiltered. This does offer an opportunity to see how a model behaves in such situations which are unfortunately, all too common. Interestingly, whether or not the model captures the false negatives depends mostly on the decoder. In addition, there are artifacts in the FCN variations that arise from the structure of the decoder itself rather than anything meaningful from the image.

<img style="display: block;
           margin-left: auto;
           margin-right: auto;
           width: 80%;"
           src="https://raw.githubusercontent.com/sig-gis/eofm-book/main/assets/masks.png"/>

<div style="display: flex; justify-content: center; gap: 20px;">
  <img style="display: block;
            margin-left: auto;
            margin-right: auto;
            width: 40%";
            src="https://raw.githubusercontent.com/sig-gis/eofm-book/main/assets/hls_label.png" />
  <img style="display: block;
            margin-left: auto;
            margin-right: auto;
            width: 40%;"
      src="https://raw.githubusercontent.com/sig-gis/eofm-book/main/assets/hls_image.png" />
</div>

<div style="text-align: center;">Figure 30: (Top) Grid table of mask outputs of all models with all decoders. (Bottom Left) Original annotation. (Bottom Right) Source Image</div>

#### Multimodality / Multisensor Models Scale

You may have noticed in spite being a small decoder, the linear probe into PRITHVI had comparable performance to the larger models in this given task. While this may be attributed to the limited training time or architectural design choices, the more likely reason is that PRITHVI 1.0 was trained exclusively on HLS scenes. Foundation Models, like all machine learning models, are shaped by their source data and making them truly generalizable is an ongoing problem. Likewise, spatiotemporal patterns are diverse in both downstream tasks and sensor data. Ultimately when selecting a base model, one of the first things to consider before benchmarking metrics is how well does the pretraining dataset align with your given task.


### MTBS Training Commands

While it is an excellent example for learning to fine tune due to its small size and simplistic design, it may be difficult to differentiate model utility with HLS burn scar dataset. Next we'll be examining the dataset on which we performed the encoder analysis by running the exact same commands as the previous section with the dataset configuration set to mtbs. Luckily for us we've also implemented the dataloader and configurations found in ./pangaea-bench/pangaea and ./pangaea-bench/configs respectively. Since this is a much larger dataset (5 times in fact), these runs may take up to two hours to run rather than a few minutes.

In [ ]:

# CROMA

!torchrun pangaea-bench/pangaea/run.py \
    --config-name=train \
    work_dir=checkpoints \
    dataset=mtbs \
    encoder=croma_optical\
    decoder=seg_muster\
    preprocessing=seg_default \
    criterion=cross_entropy \
    task=segmentation \
    use_wandb=true \
    task.trainer.n_epochs=16 \
    task.trainer.log_interval=1 \
    task.trainer.ckpt_interval=16 \
    encoder.encoder_weights=pretrained_models/CROMA_large.pt

!torchrun pangaea-bench/pangaea/run.py \
    --config-name=train \
    work_dir=checkpoints \
    dataset=mtbs \
    encoder=croma_optical\
    decoder=seg_fcn\
    preprocessing=seg_default \
    criterion=cross_entropy \
    task=segmentation \
    use_wandb=true \
    task.trainer.n_epochs=16 \
    task.trainer.log_interval=1 \
    task.trainer.ckpt_interval=16 \
    encoder.encoder_weights=pretrained_models/CROMA_large.pt

!torchrun pangaea-bench/pangaea/run.py \
    --config-name=train \
    work_dir=checkpoints \
    dataset=mtbs \
    encoder=croma_optical\
    decoder=seg_linear\
    preprocessing=seg_default \
    criterion=cross_entropy \
    task=segmentation \
    use_wandb=true \
    task.trainer.n_epochs=16 \
    task.trainer.log_interval=1 \
    task.trainer.ckpt_interval=16 \
    encoder.encoder_weights=pretrained_models/CROMA_large.pt

!torchrun pangaea-bench/pangaea/run.py \
    --config-name=train \
    work_dir=checkpoints \
    dataset=mtbs \
    encoder=croma_optical\
    decoder=seg_segformer\
    preprocessing=seg_default \
    criterion=cross_entropy \
    task=segmentation \
    use_wandb=true \
    task.trainer.n_epochs=16 \
    task.trainer.log_interval=1 \
    task.trainer.ckpt_interval=16 \
    encoder.encoder_weights=pretrained_models/CROMA_large.pt

!torchrun pangaea-bench/pangaea/run.py \
    --config-name=train \
    work_dir=checkpoints \
    dataset=mtbs \
    encoder=croma_optical\
    decoder=seg_upernet\
    preprocessing=seg_default \
    criterion=cross_entropy \
    task=segmentation \
    use_wandb=true \
    task.trainer.n_epochs=16 \
    task.trainer.log_interval=1 \
    task.trainer.ckpt_interval=16 \
    encoder.encoder_weights=pretrained_models/CROMA_large.pt

## PRITHVI

!torchrun pangaea-bench/pangaea/run.py \
    --config-name=train \
    work_dir=checkpoints \
    dataset=mtbs \
    encoder=prithvi\
    decoder=seg_muster\
    preprocessing=seg_default \
    criterion=cross_entropy \
    task=segmentation \
    use_wandb=true \
    task.trainer.n_epochs=16 \
    task.trainer.log_interval=1 \
    task.trainer.ckpt_interval=16 \
    encoder.encoder_weights=pretrained_models/Prithvi_EO_V1_100M.pt

!torchrun pangaea-bench/pangaea/run.py \
    --config-name=train \
    work_dir=checkpoints \
    dataset=mtbs \
    encoder=prithvi\
    decoder=seg_fcn\
    preprocessing=seg_default \
    criterion=cross_entropy \
    task=segmentation \
    use_wandb=true \
    task.trainer.n_epochs=16 \
    task.trainer.log_interval=1 \
    task.trainer.ckpt_interval=16 \
    encoder.encoder_weights=pretrained_models/Prithvi_EO_V1_100M.pt

!torchrun pangaea-bench/pangaea/run.py \
    --config-name=train \
    work_dir=checkpoints \
    dataset=mtbs \
    encoder=prithvi\
    decoder=seg_linear\
    preprocessing=seg_default \
    criterion=cross_entropy \
    task=segmentation \
    use_wandb=true \
    task.trainer.n_epochs=16 \
    task.trainer.log_interval=1 \
    task.trainer.ckpt_interval=16 \
    encoder.encoder_weights=pretrained_models/Prithvi_EO_V1_100M.pt

!torchrun pangaea-bench/pangaea/run.py \
    --config-name=train \
    work_dir=checkpoints \
    dataset=mtbs \
    encoder=prithvi\
    decoder=seg_segformer\
    preprocessing=seg_default \
    criterion=cross_entropy \
    task=segmentation \
    use_wandb=true \
    task.trainer.n_epochs=16 \
    task.trainer.log_interval=1 \
    task.trainer.ckpt_interval=16 \
    encoder.encoder_weights=pretrained_models/Prithvi_EO_V1_100M.pt

!torchrun pangaea-bench/pangaea/run.py \
    --config-name=train \
    work_dir=checkpoints \
    dataset=mtbs \
    encoder=prithvi\
    decoder=seg_upernet\
    preprocessing=seg_default \
    criterion=cross_entropy \
    task=segmentation \
    use_wandb=true \
    task.trainer.n_epochs=16 \
    task.trainer.log_interval=1 \
    task.trainer.ckpt_interval=16 \
    encoder.encoder_weights=pretrained_models/Prithvi_EO_V1_100M.pt

## DOFA

!torchrun pangaea-bench/pangaea/run.py \
    --config-name=train \
    work_dir=checkpoints \
    dataset=mtbs \
    encoder=dofa \
    decoder=seg_muster\
    preprocessing=seg_default \
    criterion=cross_entropy \
    task=segmentation \
    use_wandb=true \
    task.trainer.n_epochs=16 \
    task.trainer.log_interval=1 \
    task.trainer.ckpt_interval=16 \
    encoder.encoder_weights=pretrained_models/DOFA_ViT_base_e100.pth

!torchrun pangaea-bench/pangaea/run.py \
    --config-name=train \
    work_dir=checkpoints \
    dataset=dofa \
    encoder=prithvi\
    decoder=seg_fcn\
    preprocessing=seg_default \
    criterion=cross_entropy \
    task=segmentation \
    use_wandb=true \
    task.trainer.n_epochs=16 \
    task.trainer.log_interval=1 \
    task.trainer.ckpt_interval=16 \
    encoder.encoder_weights=pretrained_models/DOFA_ViT_base_e100.pth

!torchrun pangaea-bench/pangaea/run.py \
    --config-name=train \
    work_dir=checkpoints \
    dataset=dofa \
    encoder=prithvi\
    decoder=seg_linear\
    preprocessing=seg_default \
    criterion=cross_entropy \
    task=segmentation \
    use_wandb=true \
    task.trainer.n_epochs=16 \
    task.trainer.log_interval=1 \
    task.trainer.ckpt_interval=16 \
    encoder.encoder_weights=pretrained_models/DOFA_ViT_base_e100.pth

!torchrun pangaea-bench/pangaea/run.py \
    --config-name=train \
    work_dir=checkpoints \
    dataset=dofa \
    encoder=prithvi\
    decoder=seg_segformer\
    preprocessing=seg_default \
    criterion=cross_entropy \
    task=segmentation \
    use_wandb=true \
    task.trainer.n_epochs=16 \
    task.trainer.log_interval=1 \
    task.trainer.ckpt_interval=16 \
    encoder.encoder_weights=pretrained_models/DOFA_ViT_base_e100.pth

!torchrun pangaea-bench/pangaea/run.py \
    --config-name=train \
    work_dir=checkpoints \
    dataset=dofa \
    encoder=prithvi\
    decoder=seg_upernet\
    preprocessing=seg_default \
    criterion=cross_entropy \
    task=segmentation \
    use_wandb=true \
    task.trainer.n_epochs=16 \
    task.trainer.log_interval=1 \
    task.trainer.ckpt_interval=16 \
    encoder.encoder_weights=pretrained_models/DOFA_ViT_base_e100.pth

### MTBS Results

#### Overall Metrics

Unlike the previous task, in Figure 31, we see markedly poorer results. Using the typical hyperparameter settings a practitioner would apply in their first pass (i.e cross entropy loss, moderate learning rate, etc) we find that this is insufficient in producing meaningful results. Understandably, the burn intensity dataset has a greater number of classes, an additional temporal reasoning component, as well as regional differences that are not depicted here. Burn severity labels themselves are highly contextual, with the evaluation for each fire and each region being specific to those contexts and creating a very difficult challenge for all of our models. These results are also consistent with the embedding analysis in Figure 17 where we see limited separation between classes. Clever fine-tuning techniques may be able compensate for these shortcomings, however, would potentially negate any pretraining benefits and even worse, may not achieve comparable results to an end to end trained network.

<img style="display: block;
            margin-left: auto;
            margin-right: auto;
            width: 100%;"
            src="https://raw.githubusercontent.com/sig-gis/eofm-book/main/assets/mtbs_test_metrics.png"/>

<div style="text-align: center;">Figure 31: Best metrics per best decoder with each encoder.</div>

#### Interpretation

Change detection / attribution is a difficult topic, as even the veterans of the remote sensing world will tell you. It is often multivariate and involves very subtle landscape dynamics which are not clear to the human eye without aid. For example Figure 32 shows a sample pulled from the MTBS datset. These are scenes from the Boulder Lake, WA Fire in August 2022. Between the second and third image, it is clear where the fire had occured (vegetation turned to bare soil) in spite of the smoke other artifacts in the third image. Given our results from the first fine-tuning workflow, detecting this would easily done by all models. Yet when we extend the problem to intensity, the model must additionally reason what vegetation was present prior to the fire, what vegetation died, and also what vegetation experienced a low enough fire tempterature to survive and regrow the following year all while contending to the strong noisy signals ever present in all images and the variations between them. An extremely complex task.

<div style="display: flex; justify-content: center; gap: 10px;">
  <img style="display: block;
            margin-left: auto;
            margin-right: auto;
            width: 20%;"
            src="https://raw.githubusercontent.com/sig-gis/eofm-book/main/assets/boulderlake1.png"/>
  <img style="display: block;
            margin-left: auto;
            margin-right: auto;
            width: 20%; "
            src="https://raw.githubusercontent.com/sig-gis/eofm-book/main/assets/boulderlake2.png"/>
  <img style="display: block;
            margin-left: auto;
            margin-right: auto;
            width: 20%;"
            src="https://raw.githubusercontent.com/sig-gis/eofm-book/main/assets/boulderlake3.png"/>
  <img style="display: block;
            margin-left: auto;
            margin-right: auto;
            width: 20%;"
            src="https://raw.githubusercontent.com/sig-gis/eofm-book/main/assets/boulderlake4.png"/>
</div>

<div style="text-align: center;">(a)</div>

<img style="display: block;
           margin-left: auto;
           margin-right: auto;
           width: 20%;"
           src="https://raw.githubusercontent.com/sig-gis/eofm-book/main/assets/boulderlake4wlabels.png"/>

<div style="text-align: center;">(b)</div>

<div style="text-align: center;">Figure 32: (a) Landsat imagery from the Boulder Lake, WA fire in August 2022. from Left to right: 1 year prefire,  2 months post fire, 1 year post fire, 2 year post fire. (b) 2 year post fire with class labels overlaid.</div>


## A Well Defined Problem

Benchmarks such as ImageNet for vision tasks or GLUE for language tasks have helped standardize comparisons, but emerging domains may lack such comprehensive baselines. In such cases, creating task-specific benchmarks becomes more necessary especially when these "Foundation Models" become more and more prominent. Like all standardized evaluations, benchmarks can often promote excessive metric fixation. Even so, these metrics are necessary when comparing across models and must be done under the same conditions: using the same dataset splits, preprocessing steps, and evaluation protocols. Differences in any of these factors can distort the interpretation of results. For that reason, we see testing in multiple controlled conditions an essential step for a robust evaluation. In our simplified example above, we vary the decoder attached to each frozen pretrained encoder as the quality of the internal representation is only as good as how easily we can extract from it. Model complexity, training time, and inference speed are also crucial metrics to consider, especially in deployment scenarios where computational resources or latency requirements are constrained. Lastly, an assessment of representation quality is all too important to determine whether the architecture, pretraining task, or both contribute overall to model's performance. **While this process may seem overwhelming and extensive at first, we hope that you can use these guidelines to develop a strong practice of intensive validation and model evaluation, which will, in turn, provide larger benefits to your work and area of interest. It is imperative that this level of care and rigor goes into the evaluation of representation and performance to ensure that these function approximators are actually aiding in scientific discovery and other downstream tasks, and not just creating additional new webs of questions to disentangle. This should not be seen as an unpassable barrier, but a challenge and a standard that will allow us all to work together to improve capabilities in scientific undersanding and geospatial intelligence.**


## References

1) Alain, G., & Bengio, Y. (2016, October 5). Understanding intermediate layers using linear classifier probes. arXiv.org. https://arxiv.org/abs/1610.01644

2) Caron, M., Touvron, H., Misra, I., Jégou, H., Mairal, J., Bojanowski, P., & Joulin, A. (2021, April 29). Emerging Properties in Self-Supervised Vision Transformers. arXiv.org. https://arxiv.org/abs/2104.14294

3) Chen, T., Kornblith, S., Norouzi, M., & Hinton, G. (2020, February 13). A simple framework for contrastive learning of visual representations. arXiv.org. https://arxiv.org/abs/2002.05709

4) Cocke, A. E., Fulé, P. Z., & Crouse, J. E. (2005). Comparison of burn severity assessments using Differenced Normalized Burn Ratio and ground data. International Journal of Wildland Fire, 14(2), 189–198. https://doi.org/10.1071/wf04010  

5) DeMilt, R. P., LaHaye, N., & Tenneson, K. (2026). The View From Space: Navigating Instrumentation Differences with EOFMs. Machine Learning and the Physical Sciences Workshop @ NeurIPS 2025. Machine Learning and the Physical Sciences Workshop @ NeurIPS 2025. https://openreview.net/forum?id=qqwMYRjgbQ

6) Duderstadt, B., Helm, H. S., & Priebe, C. E. (2023, May 9). Comparing Foundation Models using Data Kernels. arXiv.org. https://arxiv.org/abs/2305.05126

7) Grill, J., Strub, F., Altché, F., Tallec, C., Richemond, P. H., Buchatskaya, E., Doersch, C., Pires, B. A., Guo, Z. D., Azar, M. G., Piot, B., Kavukcuoglu, K., Munos, R., & Valko, M. (2020, June 13). Bootstrap your own latent: A new approach to self-supervised Learning. arXiv.org. https://arxiv.org/abs/2006.07733

8) Han K, Wang Y, Chen H, Chen X, Guo J, Liu Z, Tang Y, Xiao A, Xu C, Xu Y, Yang Z, Zhang Y, Tao D. A Survey on Vision Transformer. IEEE Trans Pattern Anal Mach Intell. 2023 Jan;45(1):87-110. doi: 10.1109/TPAMI.2022.3152247. Epub 2022 Dec 5. PMID: 35180075.

9) He, K., Fan, H.,  Wu, Y., Xie, S. and Girshick, R., "Momentum Contrast for Unsupervised Visual Representation Learning," 2020 IEEE/CVF Conference on Computer Vision and Pattern Recognition (CVPR), Seattle, WA, USA, 2020, pp. 9726-9735, doi: 10.1109/CVPR42600.2020.00975.

10) He, K., Chen, X., Xie, S., Li, Y., Dollár, P. and Girshick, R.,  "Masked Autoencoders Are Scalable Vision Learners," 2022 IEEE/CVF Conference on Computer Vision and Pattern Recognition (CVPR), New Orleans, LA, USA, 2022, pp. 15979-15988, doi: 10.1109/CVPR52688.2022.01553.

11) Jakubik, J., Roy, S., Phillips, C. E., Fraccaro, P., Godwin, D., Zadrozny, B., Szwarcman, D., Gomes, C., Nyirjesy, G., Edwards, B., Kimura, D., Simumba, N., Chu, L., Mukkavilli, S. K., Lambhate, D., Das, K., Bangalore, R., Oliveira, D., Muszynski, M., . . . Ramachandran, R. (2023b, October 28). Foundation Models for Generalist Geospatial Artificial Intelligence. arXiv.org. https://arxiv.org/abs/2310.18660

12) LaHaye, N., Garay, M. J., Bue, B. D., El-Askary, H., & Linstead, E. (2021). A Quantitative Validation of Multi-Modal Image Fusion and Segmentation for Object Detection and Tracking. Remote Sensing, 13(12), 2364. https://doi.org/10.3390/rs13122364

13) Long, J., Shelhamer, E., & Darrell, T. (2014, November 14). Fully convolutional networks for semantic segmentation. arXiv.org. https://arxiv.org/abs/1411.4038

14) Marsocci, V., Jia, Y., Bellier, G. L., Kerekes, D., Zeng, L., Hafner, S., Gerard, S., Brune, E., Yadav, R., Shibli, A., Fang, H., Ban, Y., Vergauwen, M., Audebert, N., & Nascetti, A. (2024, December 5). PANGAEA: a global and inclusive benchmark for Geospatial foundation models. arXiv.org. https://arxiv.org/abs/2412.04204

15) Martin, Charles H.,  Mahoney, Michael W.; Implicit Self-Regularization in Deep Neural Networks: Evidence from Random Matrix Theory and Implications for Learning. JMLR 22(165):1−73, 2021

16) Martin, Charles H.,  Peng, Tongsu (Serena)  & Mahoney, Michael W. ; Predicting trends in the quality of state-of-the-art neural networks without access to training or testing data. Nature Communications 12(4122), 2021

17) McInnes, Leland, Healy, John, & Melville, James. "UMAP: Uniform Manifold Approximation and Projection for Dimension Reduction." arXiv preprint arXiv:1802.03426, 2020

18) Parajuli, P., Shinde, R., Gurung, I., Maskey, M., & Ramachandran, R. (2024). Curating AI-Ready datasets for equity and Environmental Justice: A Data-Centric AI case study. IGARSS 2022 - 2022 IEEE International Geoscience and Remote Sensing Symposium, 483–487. https://doi.org/10.1109/igarss53475.2024.10641786

19) Patterson, D., Gonzalez, J., Le, Q., Liang, C., Munguia, L., Rothchild, D., So, D., Texier, M., & Dean, J. (2021, April 21). Carbon emissions and large neural network training. arXiv.org. https://arxiv.org/abs/2104.10350

20) Phillips, Christopher and Roy, Sujit and Ankur, Kumar and Ramachandran, Rahul. (2023, August). HLS Foundation Burnscars Dataset. https://huggingface.co/ibm-nasa-geospatial/hls_burn_scars

21) Roy, D. P., Boschetti, L., & Trigg, S. N. (2006). Remote Sensing of Fire Severity: Assessing the Performance of the Normalized Burn Ratio. IEEE Geoscience and Remote Sensing Letters, 3(1), 112-116. https://doi.org/10.1109/lgrs.2005.858485

22) Strubell, E., Ganesh, A., & McCallum, A. (2019, June 5). Energy and policy considerations for deep learning in NLP. arXiv.org. https://arxiv.org/abs/1906.02243

23) Xiao, T., Liu, Y., Zhou, B., Jiang, Y., & Sun, J. (2018, July 26). Unified perceptual parsing for scene understanding. arXiv.org. https://arxiv.org/abs/1807.10221

24) Xie, E., Wang, W., Yu, Z., Anandkumar, A., Alvarez, J. M., & Luo, P. (2021, May 31). SegFormer: Simple and Efficient Design for Semantic Segmentation with Transformers. arXiv.org. https://arxiv.org/abs/2105.15203

25) Xiong, Z., Wang, Y., Zhang, F., Stewart, A. J., Hanna, J., Borth, D., Papoutsis, I., Saux, B. L., Camps-Valls, G., & Zhu, X. X. (2024, March 22). Neural Plasticity-Inspired multimodal foundation model for earth observation. arXiv.org. https://arxiv.org/abs/2403.15356

26) Xu, J., Shi, W., Gao, P., Wang, Z., & Li, Q. (2022, November 25). MUSTER: a multi-scale transformer-based decoder for semantic segmentation. arXiv.org. https://arxiv.org/abs/2211.13928v2

